# 💧 JalNetra — India River Water-Quality Platform (all rivers, all years)
**Smart India Hackathon · PS 25166**

**Upload your yearly CPCB PDFs** and run top to bottom. One notebook, **every river and every station** in the reports:

| Part | What you get |
|---|---|
| 1–3 | Reads every PDF (≈650–1,600 stations per year), reports exactly what could not be read |
| 4–5 | Clean table + WQI, CPCB class (A–E), BOD priority (I–V) for every station-year |
| 6 | India-wide overview and year-by-year trends |
| 7 | Station coordinates (verified file → curated → optional geocoding) |
| 8 | **River engine for every river**: ordered stations, river profile, multi-year comparison, map, interpolation at any lat/long |
| 9 | **Interactive dashboard**: river explorer · any station · any lat/long |
| 10 | **Downloadable reports** (PDF per station / river, Excel for researchers) |
| 11 | **Website data pack** (`web/` folder: data.json + ready-made front end) |

## 1 · Setup

In [ ]:
!pip -q install pdfplumber geopy folium plotly ipywidgets networkx reportlab openpyxl

In [ ]:
import os, re, io, glob, math, json, time, warnings, difflib, itertools
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import folium
from folium import plugins
import networkx as nx
import pdfplumber
from IPython.display import display, HTML, clear_output
import ipywidgets as W
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60, "display.width", 220, "display.max_colwidth", 80)
try:
    from google.colab import output as _co
    _co.enable_custom_widget_manager()
except Exception:
    pass

# ---------------- settings you may change ----------------
PDF_DIR = "wq_pdfs"                 # PDFs are looked for here, in the current folder and in /content
USE_OSM_RIVER_GEOMETRY = True       # draw/measure along the real river (needs internet; falls back automatically)
GEOCODE_STATIONS = True             # locate every station from its name (Nominatim, 1 request/second, ~30-40 min once; biggest rivers first; results cached in station_coords_cache.csv). You can press Stop at any time and re-run: progress is kept.
GEOCODE_YEARS = "latest"            # "latest" = only stations reported in the newest year, or "all"
GEOCODE_LIMIT = None                # e.g. 200 to test with a few stations first
OSM_RIVERS = ["Sabarmati"]          # rivers whose real river line is downloaded from OpenStreetMap (others use straight lines between stations)
DEFAULT_RIVER = "Sabarmati"
nan = np.nan

# Map tiles that work in Colab WITHOUT a key (OpenStreetMap and Carto do not).
ESRI = {
    "Esri Street": "https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    "Esri Light Gray": "https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    "Esri Topo": "https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
    "Satellite": "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
}
def base_map(location, zoom, default="Esri Light Gray"):
    """Folium map with key-free Esri tiles only (no OSM/Carto → no 403 / 'API KEY REQUIRED')."""
    m = folium.Map(location=location, zoom_start=zoom, tiles=None, control_scale=True)
    for name, url in ESRI.items():
        folium.TileLayer(tiles=url, attr="Tiles © Esri", name=name, overlay=False, control=True, show=(name == default), max_zoom=18).add_to(m)
    return m

## 2 · Find / upload the PDFs

In [ ]:
def year_of(path):
    m = re.findall(r"(?<!\d)(20[0-3]\d)(?!\d)", os.path.basename(path))
    return int(m[-1]) if m else None

def find_pdfs():
    cands = set(glob.glob(f"{PDF_DIR}/*.pdf") + glob.glob("*.pdf") + glob.glob("/content/*.pdf") + glob.glob("/content/drive/MyDrive/*.pdf"))
    out = {}
    for p in sorted(cands):
        b = os.path.basename(p).lower()
        if any(x in b for x in ("jalnetra", "sih")) or year_of(p) is None:
            continue
        out.setdefault(year_of(p), p)
    return out

os.makedirs(PDF_DIR, exist_ok=True)
PDFS = find_pdfs()
if not PDFS:
    try:
        from google.colab import files
        print("Select ALL yearly PDFs (Ctrl/Cmd-click to select several):")
        up = files.upload()
        for name, data in up.items():
            open(os.path.join(PDF_DIR, name), "wb").write(data)
        PDFS = find_pdfs()
    except ImportError:
        pass
print("PDFs found:")
for y, p in sorted(PDFS.items()):
    print(f"  {y}: {p}  ({os.path.getsize(p)/1024:.0f} KB)")
missing = [y for y in range(min(PDFS), max(PDFS) + 1) if y not in PDFS] if PDFS else []
if missing: print("⚠️ No file for year(s):", missing, "→ trends will have a gap there.")

## 3 · Read the PDFs
Tested on the real 2017, 2019, 2020 and 2024 PDFs (≈650 / 1040 / 1000 / 1600 stations). Two methods run on every file and are merged:
1. **Position method (primary)** – every word's x/y position is used: a station's name and state wrap onto lines above/below its number line, so each text line is attached to the nearest number line and split into *name* and *State* by column position.
2. **Table method (backup)** – `pdfplumber` table extraction, row by row; used only for station codes the position method missed.

A row is accepted only with a station code and **16 values** (older reports, no Fecal Streptococci) or **18 values** (9 parameters × Min/Max). `BDL` (below detection) is read as 0 for dissolved oxygen and as the reports' own floor value for BOD (1), nitrate (0.3) and coliforms (2). Everything else goes to a **rejects** table so nothing is silently lost. Parsing the big PDFs takes ~4 min; the result is cached in `parsed_cache.csv`.

In [ ]:
STATES = ["ANDHRA PRADESH","ARUNACHAL PRADESH","ASSAM","BIHAR","CHHATTISGARH","GOA","GUJARAT","HARYANA","HIMACHAL PRADESH","JHARKHAND","KARNATAKA","KERALA",
          "MADHYA PRADESH","MAHARASHTRA","MANIPUR","MEGHALAYA","MIZORAM","NAGALAND","ODISHA","ORISSA","PUNJAB","RAJASTHAN","SIKKIM","TAMIL NADU","TELANGANA","TRIPURA",
          "UTTAR PRADESH","UTTARAKHAND","UTTARANCHAL","WEST BENGAL","DELHI","JAMMU & KASHMIR","JAMMU AND KASHMIR","JAMMU KASHMIR","J&K","LADAKH","PUDUCHERRY","PONDICHERRY",
          "CHANDIGARH","DADRA & NAGAR HAVELI","DADRA AND NAGAR HAVELI","DAMAN & DIU","DAMAN AND DIU","DNH & DD","ANDAMAN & NICOBAR","LAKSHADWEEP","INTER-STATE","INTERSTATE"]
STATE_CANON = {"ORISSA": "ODISHA", "UTTARANCHAL": "UTTARAKHAND", "PONDICHERRY": "PUDUCHERRY", "JAMMU AND KASHMIR": "JAMMU & KASHMIR", "JAMMU KASHMIR": "JAMMU & KASHMIR", "J&K": "JAMMU & KASHMIR"}
_alpha = lambda s: re.sub(r"[^A-Z]", "", s.upper())
STATE_KEYS = sorted(((_alpha(s), s) for s in STATES), key=lambda x: -len(x[0]))

def split_state(head):
    """'RIVER X AT Y CHHATTISGAR H' -> ('RIVER X AT Y', 'CHHATTISGARH'). The state is the last words of the text cell, even if split by the PDF."""
    head = re.sub(r"\s+", " ", head).strip()
    letters = _alpha(head)
    for key, s in STATE_KEYS:
        if letters.endswith(key):
            need, i = len(key), len(head)
            while need > 0 and i > 0:
                i -= 1
                if head[i].isalpha(): need -= 1
            return head[:i].strip(" ,;-("), STATE_CANON.get(s, s)
    return head, ""

NULL_TOK = {"-", "--", "—", "–", "NA", "N/A", "ND", "BDL", "NIL", "NR", "NM", "NIL.", ""}
NUM_RE = re.compile(r"^[<>~≤≥]?\s*-?\d+(?:[.,]\d+)?\*?$")
is_numtok = lambda t: (t.strip().upper() in NULL_TOK) or bool(NUM_RE.match(t.strip()))
def tok_to_num(t):
    t = t.strip()
    if t.upper() in NULL_TOK: return nan
    t = re.sub(r"[<>~≤≥*\s]", "", t)
    t = t.replace(",", "") if re.fullmatch(r"\d{1,3}(,\d{3})+", t) else t.replace(",", ".")
    try: return float(t)
    except ValueError: return nan

PARAM_COLS = [f"{p}_{s}" for p in ["Temp","DO","pH","Conductivity","BOD","Nitrate","Fecal_Coliform","Total_Coliform","Fecal_Streptococci"] for s in ("Min","Max")]

TITLE_RIVER = re.compile(r"water\s+quality\s+of\s+(?:the\s+)?rivers?\s+([A-Za-z][A-Za-z .&'/-]{1,40}?)\s*(?:\(|[-–:]|\b20\d\d\b|$)", re.I)
TITLE_TRIB = re.compile(r"water\s+quality\s+of\s+(?:tributar|minor|other)", re.I)

def page_river(text, current):
    """River named in a table title ('Table 1: Water Quality of river Beas - 2024'); carried forward across pages."""
    for l in (text or "").split("\n")[:12]:
        if TITLE_TRIB.search(l): return None
        m = TITLE_RIVER.search(l)
        if m: return m.group(1).strip(" .-").title()
    return current

def split_cells(cells):
    out = []
    for c in cells:
        c = re.sub(r"\s+", " ", (c or "").replace("\n", " ")).strip()
        if not c: continue
        parts = c.split(" ")
        out.extend(parts if len(parts) > 1 and all(is_numtok(p) for p in parts) else [c])
    return out

BDL_VALUE = {8: 1.0, 9: 1.0, 10: 0.3, 11: 0.3, 12: 2.0, 13: 2.0, 14: 2.0, 15: 2.0, 16: 2.0, 17: 2.0}   # BOD 1 mg/L, nitrate 0.3 mg/L, coliforms 2 MPN = the floor values the reports print for "below detection"
def values_from_tail(nums):
    """16 values = no Fecal Streptococci column (older reports); 18 = full."""
    if len(nums) == 16: nums = nums + ["-", "-"]
    if len(nums) != 18: return None
    v = [tok_to_num(x) for x in nums]
    for i, t in enumerate(nums):
        if t.strip().upper() in ("BDL", "NIL", "ND", "NIL.", "<DL"):
            if i in (2, 3): v[i] = 0.0                      # DO below detection = no measurable oxygen
            else: v[i] = BDL_VALUE.get(i, nan)              # below detection limit -> the reports' own floor value
    return v

def parse_tables(pdf):
    rows, rej, river = [], [], None
    for pno, page in enumerate(pdf.pages, 1):
        river = page_river(page.extract_text(), river)
        try: tables = page.extract_tables() or []
        except Exception: tables = []
        for t in tables:
            for r in t:
                ne = split_cells(r)
                if len(ne) < 18 or not re.fullmatch(r"\d{1,6}", ne[0]): continue
                k = len(ne)
                while k > 1 and is_numtok(ne[k - 1]): k -= 1
                vals = values_from_tail(ne[k:])
                if vals is None or k < 2:
                    rej.append({"page": pno, "reason": f"{len(ne) - k} value cells (need 16 or 18)", "text": " | ".join(ne)[:200]}); continue
                name, state = split_state(" ".join(ne[1:k]))
                rows.append({"Station_Code": int(ne[0]), "Station_Name": name, "State": state, "Table_River": river, **dict(zip(PARAM_COLS, vals)), "page": pno, "method": "table"})
    return rows, rej

HEADER_KEYS = re.compile(r"(statio\s?n\b|name of the|monitoring|temperature|conductivity|coliform|streptococci|nitrate|location)", re.I)
HEADER_STRONG = re.compile(r"(mpn|μmho|umho|mi\s?n\s+ma\s?x|min\s+max|water quality|^\s*page\s*\d|mg/\s?l\b|^\s*ml\b|^\s*mi\s+ma\b|^n x n x|^\s*table\b|central pollution|\bcode\b$|^\s*(location|state|station code)\s*$|moef|under e \(p\)|rules,? 19\d\d)", re.I)
def is_header_line(l):
    return bool(HEADER_STRONG.search(l)) or len(set(m.lower() for m in HEADER_KEYS.findall(l))) >= 2

STATE_VOCAB = {_alpha(w) for st_ in STATES for w in st_.replace("&", " ").split()} | {"CHHATTISGAR", "UTTARAKHAN", "KARNATAK", "MAHARASHTR", "RAJASTHA"}
SHORT_NAME_WORDS = {"AT", "OF", "TO", "ON", "IN", "DS", "US", "D/S", "U/S", "NR", "NH", "KM", "R", "B/C"}
def parse_text(pdf):
    """Geometry method: a station's name/state wrap onto lines above and below its number line; every text line is attached to the
    nearest number line, then name words (left of the State column) and state words (State column) are separated by x-position."""
    rows, rej, river, state_x, counter = [], [], None, None, itertools.count()
    for pno, page in enumerate(pdf.pages, 1):
        river = page_river(page.extract_text(), river)
        words = page.extract_words(x_tolerance=2, y_tolerance=2)
        if not words: continue
        hs = [w for w in words if w["text"].strip().upper() == "STATE" and w["top"] < 160]
        hl = [w for w in words if w["text"].strip().upper() in ("LOCATION", "MONITORING") and w["top"] < 160]
        if hs:
            c_state = (hs[0]["x0"] + hs[0]["x1"]) / 2
            c_name = (hl[-1]["x0"] + hl[-1]["x1"]) / 2 if hl else c_state - 90
            state_x = (c_name + c_state) / 2          # words whose CENTRE lies right of this belong to the State column (both columns are centre-aligned)
        if state_x is None: continue
        lines = []
        for w in sorted(words, key=lambda w: (round(w["top"]), w["x0"])):
            if lines and abs(lines[-1]["top"] - w["top"]) <= 2.5: lines[-1]["w"].append(w)
            else: lines.append({"top": w["top"], "w": [w]})
        mains = []
        for ln in lines:
            ws = sorted(ln["w"], key=lambda w: w["x0"]); txt = " ".join(w["text"] for w in ws)
            ln["ws"], ln["txt"] = ws, txt
            if re.fullmatch(r"\d{1,6}", ws[0]["text"]) and len(ws) >= 17:
                k = len(ws)
                while k > 1 and is_numtok(ws[k - 1]["text"]): k -= 1
                if len(ws) - k >= 16: ln["nk"] = k; mains.append(ln)
        if not mains: continue
        mine = {id(m): [] for m in mains}
        for ln in lines:
            if "nk" in ln or is_header_line(ln["txt"]) or re.fullmatch(r"\d{1,3}", ln["txt"]): continue
            near = min(mains, key=lambda m: abs(m["top"] - ln["top"]))
            if abs(near["top"] - ln["top"]) <= 40: mine[id(near)].append(ln)
        for m in mains:
            ws, k = m["ws"], m["nk"]
            vals = values_from_tail([w["text"] for w in ws[k:]])
            parts = sorted(mine[id(m)] + [{"top": m["top"], "ws": ws[1:k]}], key=lambda l: l["top"])
            ctr = lambda w: (w["x0"] + w["x1"]) / 2
            allw = [w for l in parts for w in l["ws"]]
            is_st = lambda w: ctr(w) >= state_x and (_alpha(w["text"]) in STATE_VOCAB or (len(_alpha(w["text"])) <= 2 and w["text"].isalpha() and w["text"].upper() not in SHORT_NAME_WORDS))
            nm = " ".join(w["text"] for w in allw if not is_st(w))
            st = " ".join(w["text"] for w in allw if is_st(w))
            if vals is None:
                rej.append({"page": pno, "reason": f"{len(ws) - k} value tokens (need 16 or 18)", "text": m["txt"][:200]}); continue
            left, state = split_state(st); state = state or st.upper()
            if left: nm = f"{nm} {left}"
            rows.append({"Station_Code": int(ws[0]["text"]), "Station_Name": re.sub(r"\s+", " ", nm).strip(" ,"), "State": state, "Table_River": river, **dict(zip(PARAM_COLS, vals)), "page": pno, "ord": next(counter), "method": "text"})
    return rows, rej

def parse_pdf(path, year):
    with pdfplumber.open(path) as pdf:
        n = len(pdf.pages)
        rt, jt = parse_tables(pdf)
        rx, jx = parse_text(pdf)
    A, B = pd.DataFrame(rt), pd.DataFrame(rx)
    agree = np.nan
    if len(A) and len(B):                       # geometry rows are primary (names/states split correctly); table rows only for codes it missed
        A2 = A[~A.Station_Code.isin(B.Station_Code)]
        df = pd.concat([B, A2], ignore_index=True)
    else:
        df = A if len(A) else B
    rej = pd.DataFrame(jt + jx)
    if len(rej) and len(df):          # a line rejected by one method but read by the other is not a loss
        code_of = rej.text.str.extract(r"^\s*(\d{1,6})")[0].astype(float)
        rej = rej[~code_of.isin(df.Station_Code)].reset_index(drop=True)
    rep = {"Year": year, "pages": n, "rows_table_method": len(A), "rows_text_method": len(pd.DataFrame(rx)), "rows_final": len(df),
           "unique_codes": df.Station_Code.nunique() if len(df) else 0, "rejected_lines": len(rej),
           "no_state_found": int((df.State == "").sum()) if len(df) else 0}
    if len(df): df.insert(0, "Year", year)
    if len(rej): rej.insert(0, "Year", year)
    return df, rej, rep

def debug_page(year, page_no=1, n_chars=2500):
    """Show what the PDF really contains on one page (use this if a year parses badly)."""
    with pdfplumber.open(PDFS[year]) as pdf:
        pg = pdf.pages[page_no - 1]
        print("---- TEXT ----"); print((pg.extract_text() or "")[:n_chars])
        tb = pg.extract_tables()
        print(f"---- {len(tb)} table(s) ----")
        for t in tb[:1]:
            for r in t[:6]: print(r)

USE_PARSE_CACHE = True      # parsing the big PDFs takes a few minutes; delete parsed_cache.csv to force a re-read
CACHE = "parsed_cache.csv"
all_rows, all_rej, reports = [], [], []
_cached = pd.read_csv(CACHE) if (USE_PARSE_CACHE and os.path.exists(CACHE)) else None
if _cached is not None and set(_cached.Year.unique()) >= set(PDFS):
    print("Using cached parse (delete parsed_cache.csv to re-read the PDFs).")
    all_rows = [_cached[_cached.Year == y] for y in sorted(PDFS)]; all_rej = [pd.DataFrame()]; reports = [{"Year": y, "rows_final": int((_cached.Year == y).sum())} for y in sorted(PDFS)]
for y, p in ([] if all_rows else sorted(PDFS.items())):
    t0 = time.time()
    try:
        d, j, r = parse_pdf(p, y)
    except Exception as e:
        d, j, r = pd.DataFrame(), pd.DataFrame(), {"Year": y, "rows_final": 0, "error": str(e)[:100]}
    all_rows.append(d); all_rej.append(j); reports.append(r)
    print(f"{y}: {r.get('rows_final', 0)} rows in {time.time()-t0:.0f}s" + (f"  ERROR {r.get('error')}" if 'error' in r else ""))

raw_all = pd.concat([d for d in all_rows if len(d)], ignore_index=True) if any(len(d) for d in all_rows) else pd.DataFrame()
if len(raw_all) and _cached is None: raw_all.to_csv(CACHE, index=False)
rejects = pd.concat([j for j in all_rej if len(j)], ignore_index=True) if any(len(j) for j in all_rej) else pd.DataFrame(columns=["Year", "page", "reason", "text"])
parse_report = pd.DataFrame(reports)
display(parse_report)

### ✅ Check the parse
* `rows_final` should be roughly the number of monitoring stations printed in each report (≈ 500–700 for recent years).
* Many rows in `rejected_lines` or a year with 0 rows → run `debug_page(year, 1)` and look at how the PDF text/tables really look; if the layout is unusual, send me that output and the parser can be adapted to it in minutes.
* `no_state_found` > 0 means a row was read but its state was not recognised (the row is kept, with an empty state).

In [ ]:
if len(rejects):
    print("Reasons for rejected lines:"); display(rejects.groupby(["Year", "reason"]).size().rename("lines").reset_index().head(40))
    print("Sample of rejected lines (check that nothing important is hiding here):"); display(rejects.sample(min(10, len(rejects)), random_state=1))

## 4 · Clean + validate (all years)
* impossible values (e.g. Temp 50300, pH 2 200 000) are blanked — they come from PDF columns that slid sideways
* Min > Max pairs are blanked too
* duplicates (same year + code + same values) are removed
* a **River** column is derived from the station name

In [ ]:
VALID = {"Temp": (0, 45), "DO": (0, 20), "pH": (3, 12), "Conductivity": (1, 60000), "BOD": (0, 500), "Nitrate": (0, 200),
         "Fecal_Coliform": (0, 1e7), "Total_Coliform": (0, 1e8), "Fecal_Streptococci": (0, 1e7)}

def sanity_clean(df):
    d = df.copy(); rep = []
    d["n_flagged"] = 0
    for p, (lo, hi) in VALID.items():
        for s in ("Min", "Max"):
            c = f"{p}_{s}"
            bad = (d[c] < lo) | (d[c] > hi)
            d.loc[bad, c] = nan; d.loc[bad, "n_flagged"] += 1
            rep.append((c, int(bad.sum())))
        swap = d[f"{p}_Min"] > d[f"{p}_Max"]
        d.loc[swap, [f"{p}_Min", f"{p}_Max"]] = nan; d.loc[swap, "n_flagged"] += 1
        rep.append((f"{p} (min>max)", int(swap.sum())))
    return d, pd.DataFrame(rep, columns=["check", "values_blanked"])

def river_of(name):
    s = re.sub(r"\s+", " ", str(name).upper()).strip()
    s = re.sub(r"^(RIVER|R\.|NALA|NALLAH)[\s,]+", "", s)
    m = re.match(r"^([A-Z][A-Z .\-']{2,30}?)(?:\s+(?:AT|AFTER|BEFORE|D/S|U/S|NEAR|DOWN|UP|ON|FROM|BELOW|ABOVE|NALLAH|NALA|\(|,|-|/|\d)|$)", s)
    return (m.group(1).strip(" .-") if m else s.split(" AT ")[0][:30]).title()

ALIAS = {"SATLUJ": "SUTLEJ", "GANGES": "GANGA", "JAMUNA": "YAMUNA", "KAVERI": "CAUVERY", "KAVERY": "CAUVERY", "NARBADA": "NARMADA", "SABARMAATI": "SABARMATI", "SABERMATI": "SABARMATI",
         "BRAHMPUTRA": "BRAHMAPUTRA", "GODAVARY": "GODAVARI", "TUNGBHADRA": "TUNGABHADRA", "TAPTI": "TAPI", "BAITARNI": "BAITARANI", "MAHANADDI": "MAHANADI", "KRISHNAA": "KRISHNA"}
def canon_rivers(names):
    """Merge spelling variants (Satluj/Sutlej, Kaveri/Cauvery …) and fuzzy-match rare spellings onto well-known names."""
    key = names.fillna("").astype(str).map(lambda x: ALIAS.get(re.sub(r"[^A-Z]", "", x.upper()), re.sub(r"[^A-Z]", "", x.upper())))
    counts = key.value_counts(); big = [k for k, c in counts.items() if c >= 6 and k]
    def fz(k):
        if not k or counts[k] >= 6: return k
        m = difflib.get_close_matches(k, big, n=1, cutoff=0.87); return m[0] if m else k
    key = key.map(fz)
    label = {k: names[key == k].mode().iloc[0] for k in key.unique()}
    label = {k: ({"SUTLEJ": "Sutlej", "GANGA": "Ganga", "YAMUNA": "Yamuna", "CAUVERY": "Cauvery", "NARMADA": "Narmada", "SABARMATI": "Sabarmati"}.get(k, v)) for k, v in label.items()}
    return key.map(label)

if len(raw_all):
    df, qa = sanity_clean(raw_all)
    valid_states = {STATE_CANON.get(x, x) for x in STATES} | {""}
    df.loc[~df.State.isin(valid_states), "State"] = ""            # junk from mis-split columns
    df["Station_Name"] = df.Station_Name.str.replace(r"\s+", " ", regex=True).str.strip(" ,")
    df["River"] = canon_rivers(df.Station_Name.map(river_of))
    _n = df.groupby("River").Station_Code.nunique()
    df["RiverGroup"] = np.where(df.River.map(_n) >= 3, df.River, "Other streams" + (" – " + df.State).where(df.State != "", ""))
    if "ord" not in df: df["ord"] = df.groupby("Year").cumcount()
    df["ord"] = df.ord.fillna(df.groupby("Year").ord.transform("max") + 1)
    df = df.drop_duplicates(["Year", "Station_Code"] + PARAM_COLS).reset_index(drop=True)
    # keep rows that have at least some data
    df = df[df[PARAM_COLS].notna().sum(axis=1) >= 4].reset_index(drop=True)
    print(f"Clean table: {len(df)} station-year rows, {df.Station_Code.nunique()} stations, years {sorted(df.Year.unique())}")
    display(qa[qa.values_blanked > 0])
    cov = df.groupby("Year")[PARAM_COLS[::2]].agg(lambda s: s.notna().mean()).round(2)
    cov.columns = [c.replace("_Min", "") for c in cov.columns]
    print("Share of stations with each parameter, per year (low values = parameter not reported that year):"); display(cov)
else:
    raise SystemExit("No data parsed — see the parse report above and run debug_page(year, 1).")

## 5 · Scoring: WQI, CPCB class, BOD priority
**Two views of each station-year** (data are yearly Min/Max ranges): **worst case** (DO min, BOD/EC/Nitrate/Coliform max, pH farthest from 7) and **typical** (midpoint).

**JalNetra WQI** 0 = pristine … 100+ = severely polluted. Each parameter gets a 0–200 sub-index interpolated between CPCB class limits (value at Class-A limit → 25, Class-B limit → 50 …), combined with weights below. *This is our own transparent index, not an official CPCB one.* Edit `ANCHORS`/`WEIGHTS` to change it.

In [ ]:
ANCHORS = {
 "DO":  ([0, 2, 4, 5, 6, 8],            [200, 100, 75, 50, 25, 0]),
 "BOD": ([1, 2, 3, 6, 10, 30, 100],      [0, 25, 50, 75, 100, 150, 200]),
 "FC":  ([10, 50, 500, 2500, 5000, 50000], [0, 25, 50, 75, 100, 150]),
 "EC":  ([250, 750, 1500, 2250, 4000, 8000], [0, 25, 50, 75, 100, 150]),
 "NO3": ([1, 5, 10, 20, 45, 100],        [0, 25, 50, 75, 100, 150]),
 "pHdev": ([0.5, 1.5, 2.0, 2.5, 3.5],    [0, 50, 75, 100, 150]),
}
LOGSCALE = {"BOD", "FC", "EC", "NO3"}
WEIGHTS = {"DO": 0.25, "BOD": 0.25, "FC": 0.20, "EC": 0.10, "pH": 0.10, "NO3": 0.10}
PARAMS = ["DO", "BOD", "pH", "EC", "NO3", "FC", "TC", "Temp"]
LABELS = {"DO": "Dissolved Oxygen (mg/L)", "BOD": "BOD (mg/L)", "pH": "pH", "EC": "Conductivity (µmho/cm)", "NO3": "Nitrate+Nitrite N (mg/L)",
          "FC": "Fecal Coliform (MPN/100mL)", "TC": "Total Coliform (MPN/100mL)", "Temp": "Temperature (°C)"}
COLMAP = {"DO": "DO", "BOD": "BOD", "pH": "pH", "EC": "Conductivity", "NO3": "Nitrate", "FC": "Fecal_Coliform", "TC": "Total_Coliform", "Temp": "Temp"}
LOGP = {"BOD", "EC", "FC", "TC"}
isn = lambda x: x is None or (isinstance(x, float) and math.isnan(x))

def _sub(p, x):
    if isn(x): return nan
    xs, ys = ANCHORS[p]
    return float(np.interp(math.log10(max(x, 1e-9)), np.log10(xs), ys)) if p in LOGSCALE else float(np.interp(x, xs, ys))

def vector(row, mode="worst"):
    g = lambda p, s: row.get(f"{COLMAP[p]}_{s}", nan)
    v = {}
    if mode == "worst":
        v.update(DO=g("DO", "Min"), BOD=g("BOD", "Max"), EC=g("EC", "Max"), NO3=g("NO3", "Max"), FC=g("FC", "Max"), TC=g("TC", "Max"), Temp=g("Temp", "Max"))
        lo, hi = g("pH", "Min"), g("pH", "Max")
        v["pH"] = hi if isn(lo) else lo if isn(hi) else (lo if abs(lo - 7) > abs(hi - 7) else hi)
        v["pH_lo"], v["pH_hi"] = lo, hi
    else:
        for p in PARAMS:
            a, b = g(p, "Min"), g(p, "Max")
            v[p] = nan if (isn(a) and isn(b)) else float(np.nanmean([a, b]))
        v["pH_lo"] = v["pH_hi"] = v["pH"]
    return v

def sub_indices(v):
    return {"DO": _sub("DO", v["DO"]), "BOD": _sub("BOD", v["BOD"]), "FC": _sub("FC", v["FC"]), "EC": _sub("EC", v["EC"]), "NO3": _sub("NO3", v["NO3"]),
            "pH": nan if isn(v["pH"]) else _sub("pHdev", abs(v["pH"] - 7))}

def wqi(v):
    s = sub_indices(v); num = den = 0.0
    for p, w in WEIGHTS.items():
        if not isn(s[p]): num += w * s[p]; den += w
    return num / den if den >= 0.35 else nan          # need at least ~1/3 of the weight present

WQI_BANDS = [(25, "Excellent", "#1a9850"), (50, "Good", "#91cf60"), (75, "Poor", "#fdae61"), (100, "Very poor", "#f46d43"), (1e9, "Severely polluted", "#a50026")]
def wqi_cat(x):
    if isn(x): return ("No data", "#9e9e9e")
    for hi, name, col in WQI_BANDS:
        if x <= hi: return (name, col)

_ok = lambda val, test: True if isn(val) else bool(test(val))
CLASS_DEF = {
 "A": ("Drinking water without conventional treatment, after disinfection", lambda v: _ok(v["DO"], lambda x: x >= 6) and _ok(v["BOD"], lambda x: x <= 2) and _ok(v["TC"], lambda x: x <= 50) and _ok(v["pH_lo"], lambda x: x >= 6.5) and _ok(v["pH_hi"], lambda x: x <= 8.5)),
 "B": ("Outdoor bathing (organised)", lambda v: _ok(v["DO"], lambda x: x >= 5) and _ok(v["BOD"], lambda x: x <= 3) and _ok(v["TC"], lambda x: x <= 500) and _ok(v["pH_lo"], lambda x: x >= 6.5) and _ok(v["pH_hi"], lambda x: x <= 8.5)),
 "C": ("Drinking water after conventional treatment and disinfection", lambda v: _ok(v["DO"], lambda x: x >= 4) and _ok(v["BOD"], lambda x: x <= 3) and _ok(v["TC"], lambda x: x <= 5000) and _ok(v["pH_lo"], lambda x: x >= 6.5) and _ok(v["pH_hi"], lambda x: x <= 8.5)),
 "D": ("Propagation of wildlife and fisheries", lambda v: _ok(v["DO"], lambda x: x >= 4) and _ok(v["pH_lo"], lambda x: x >= 6.5) and _ok(v["pH_hi"], lambda x: x <= 8.5)),
 "E": ("Irrigation, industrial cooling, controlled waste disposal", lambda v: _ok(v["EC"], lambda x: x <= 2250) and _ok(v["pH_lo"], lambda x: x >= 6.0) and _ok(v["pH_hi"], lambda x: x <= 8.5)),
}
def cpcb_class(v):
    if all(isn(v[k]) for k in ("DO", "BOD", "TC", "EC", "pH")): return "No data"
    for c in "ABCDE":
        if CLASS_DEF[c][1](v): return c
    return "Below E"

PRIORITY_BANDS = [(30, "I"), (20, "II"), (10, "III"), (6, "IV"), (3, "V")]
PRIORITY_ORDER = ["I", "II", "III", "IV", "V", "Not polluted", "No data"]
PRIORITY_COL = {"I": "#a50026", "II": "#d73027", "III": "#f46d43", "IV": "#fdae61", "V": "#fee08b", "Not polluted": "#1a9850", "No data": "#9e9e9e"}
CLASS_COL = {"A": "#1a9850", "B": "#91cf60", "C": "#d9ef8b", "D": "#fdae61", "E": "#f46d43", "Below E": "#a50026", "No data": "#9e9e9e"}
def priority(bod_max):
    if isn(bod_max): return "No data"
    for thr, name in PRIORITY_BANDS:
        if bod_max > thr: return name
    return "Not polluted"

def evaluate(vw, vt=None):
    e = {"WQI_worst": wqi(vw), "CPCB_class": cpcb_class(vw), "Priority": priority(vw["BOD"])}
    e["WQI_cat"] = wqi_cat(e["WQI_worst"])[0]
    if vt is not None:
        e["WQI_typical"] = wqi(vt); e["WQI_typ_cat"] = wqi_cat(e["WQI_typical"])[0]
    return e

def score_frame(d):
    recs = []
    for _, r in d.iterrows():
        vw, vt = vector(r, "worst"), vector(r, "typical")
        sw = sub_indices(vw)
        recs.append({**{f"w_{k}": vw[k] for k in PARAMS}, **{f"t_{k}": vt[k] for k in PARAMS}, **{f"si_{k}": sw[k] for k in sw}, **evaluate(vw, vt)})
    return pd.concat([d.reset_index(drop=True), pd.DataFrame(recs)], axis=1)

scored = score_frame(df)
scored["WQI_color"] = scored.WQI_worst.map(lambda x: wqi_cat(x)[1])
YEARS = sorted(scored.Year.unique()); LATEST = YEARS[-1]
print(f"Scored {len(scored)} station-years.  Latest year: {LATEST}")
display(scored.groupby("Year").agg(stations=("Station_Code", "nunique"), median_WQI=("WQI_worst", "median"), pct_priority_I_II=("Priority", lambda s: round(100 * s.isin(["I", "II"]).mean(), 1))).round(1))

## 6 · India-wide overview and trends (all years, all rivers — no coordinates needed)

In [ ]:
CAT_ORDER = [b[1] for b in WQI_BANDS] + ["No data"]; CAT_COL = {b[1]: b[2] for b in WQI_BANDS}; CAT_COL["No data"] = "#9e9e9e"

# 6.1 how much data exists
cnt = scored.groupby("Year").agg(rows=("Station_Code", "size"), stations=("Station_Code", "nunique"), states=("State", "nunique"))
fig = make_subplots(rows=1, cols=2, subplot_titles=("Monitoring stations per year", "Parameter coverage (share of stations)"), column_widths=[0.4, 0.6])
fig.add_bar(x=cnt.index.astype(str), y=cnt.stations, marker_color="#2c7fb8", showlegend=False, row=1, col=1)
fig.add_trace(go.Heatmap(z=cov.values.T, x=cov.index.astype(str), y=cov.columns, colorscale="Blues", zmin=0, zmax=1, colorbar=dict(title="share")), row=1, col=2)
fig.update_layout(height=380, title="What the PDFs contain"); fig.show()

In [ ]:
# 6.2 water-quality categories and priority by year
def stack_share(col, order, colors, title):
    t = scored.groupby(["Year", col]).size().unstack(fill_value=0).reindex(columns=order, fill_value=0)
    t = t.div(t.sum(axis=1), axis=0) * 100
    fig = go.Figure([go.Bar(name=c, x=t.index.astype(str), y=t[c], marker_color=colors[c]) for c in order if t[c].sum() > 0])
    fig.update_layout(barmode="stack", title=title, yaxis_title="% of stations", height=380); return fig
stack_share("WQI_cat", CAT_ORDER, CAT_COL, "Share of stations by JalNetra WQI category (worst case)").show()
stack_share("Priority", PRIORITY_ORDER, PRIORITY_COL, "Share of stations by BOD priority class (I = most polluted)").show()
stack_share("CPCB_class", ["A", "B", "C", "D", "E", "Below E", "No data"], CLASS_COL, "Share of stations by CPCB best-use class").show()

In [ ]:
# 6.3 parameter distributions over the years
fig = make_subplots(rows=2, cols=3, subplot_titles=["DO min (mg/L)", "BOD max (mg/L, log)", "Conductivity max (log)", "Fecal coliform max (log)", "Nitrate max (mg/L, log)", "pH max"])
for (r, c), col, lg in zip([(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3)], ["DO_Min", "BOD_Max", "Conductivity_Max", "Fecal_Coliform_Max", "Nitrate_Max", "pH_Max"], [0, 1, 1, 1, 1, 0]):
    for y in YEARS:
        d = scored[scored.Year == y][col].dropna()
        fig.add_trace(go.Box(y=d, name=str(y), marker_color="#4a6fa5", showlegend=False, boxpoints=False), r, c)
    if lg: fig.update_yaxes(type="log", row=r, col=c)
fig.update_layout(height=620, title="How India's river parameters moved, year by year (box = middle 50 % of stations)"); fig.show()

In [ ]:
# 6.4 states and stations
last = scored[scored.Year == LATEST]
st = last.groupby("State").agg(stations=("Station_Code", "nunique"), median_WQI=("WQI_worst", "median"), worst_BOD=("BOD_Max", "max"), n_priority_I_II=("Priority", lambda s: int(s.isin(["I", "II"]).sum()))).query("stations>=3").sort_values("median_WQI", ascending=False)
fig = px.bar(st.reset_index().head(25), x="State", y="median_WQI", color="n_priority_I_II", color_continuous_scale="Reds", hover_data=["stations", "worst_BOD"], title=f"States with the most polluted rivers, {LATEST} (median worst-case WQI; colour = # stations in Priority I–II)")
fig.update_layout(height=420, xaxis_tickangle=-40); fig.show()
top = last.sort_values("WQI_worst", ascending=False).head(20)
display(HTML(f"<h4>Top 20 most polluted stations, {LATEST}</h4>")); display(top[["Station_Code", "Station_Name", "State", "WQI_worst", "BOD_Max", "DO_Min", "Priority", "CPCB_class"]].round(1))

In [ ]:
# 6.5 who improved, who got worse (first vs last year available per station)
pv = scored.pivot_table(index="Station_Code", columns="Year", values="WQI_worst", aggfunc="mean")
names = scored.sort_values("Year").groupby("Station_Code").agg(Station_Name=("Station_Name", "last"), State=("State", "last"))
def first_last(row):
    v = row.dropna()
    return pd.Series({"first_year": v.index[0], "last_year": v.index[-1], "WQI_first": v.iloc[0], "WQI_last": v.iloc[-1]}) if len(v) >= 2 else pd.Series(dtype=float)
chg = pv.apply(first_last, axis=1).dropna()
chg["change"] = chg.WQI_last - chg.WQI_first
chg = chg.join(names)
if len(chg):
    show = pd.concat([chg.nsmallest(12, "change").assign(trend="improved"), chg.nlargest(12, "change").assign(trend="worsened")])
    fig = px.bar(show.sort_values("change"), x="change", y=show.sort_values("change").Station_Name.str[:55], color="trend", color_discrete_map={"improved": "#1a9850", "worsened": "#d73027"}, orientation="h",
                 title="Biggest WQI changes between a station's first and last reported year (negative = cleaner)", hover_data=["State", "first_year", "last_year"])
    fig.update_layout(height=620, yaxis_title=None, margin=dict(l=300)); fig.show()
    print(f"Stations with ≥2 years: {len(chg)} · improved (>5 pts): {(chg.change < -5).sum()} · worsened (>5 pts): {(chg.change > 5).sum()} · stable: {(chg.change.abs() <= 5).sum()}")

## 7 · Where are the stations? (coordinates)
The CPCB tables carry **no latitude/longitude**. Sources, in order of trust:
1. **`verified_coordinates.csv`** (columns `Station_Code, lat, lon`) — official positions, always wins
2. **curated Sabarmati sites** (approximate — replace!)
3. **geocoding** of station names (`GEOCODE_STATIONS = True`): approximate, checked to lie in the right state, shown as *approx.* everywhere

Rivers with fewer than two located stations still get every table and chart; only the map and lat/long interpolation need coordinates.

In [ ]:
SAB_COORDS = {   # approximate (lat, lon) — verify against CPCB / GPCB / WRIS and replace
 "Kheroj": (24.040, 73.020), "Dharoi Dam": (23.994, 72.859), "Chiloda": (23.210, 72.610), "Hansol": (23.079, 72.629),
 "Railway Bridge": (23.033, 72.580), "V.N. Bridge": (22.992, 72.570), "Miroli": (22.930, 72.580), "Vautha": (22.590, 72.430),
 "Meshwa @ Samlaji": (23.676, 73.349), "Shedhi @ Kheda": (22.750, 72.685), "Khari @ Lali": (22.970, 72.520)}
SITE_RULES = [(r"KHEROJ", "Kheroj"), (r"DHAROI", "Dharoi Dam"), (r"CHILODA|LEKAWADA", "Chiloda"), (r"HANSOL|HANSAOL", "Hansol"),
              (r"RAILWAY\s*BRIDGE", "Railway Bridge"), (r"V\.?\s?N\.?\s*BRIDGE", "V.N. Bridge"), (r"MIROLI", "Miroli"), (r"VAUTHA|DHOKLA", "Vautha"),
              (r"MESHW[AO]", "Meshwa @ Samlaji"), (r"SHEDHI", "Shedhi @ Kheda"), (r"KHARI", "Khari @ Lali")]
SABAR = r"SABARM+A+TI|SABERMATI"
def curated_site(name, state):
    if state not in ("GUJARAT", "RAJASTHAN", ""): return None
    if not re.search(SABAR + r"|MESHW[AO]|SHEDHI|KHARI", name, re.I): return None
    for pat, site in SITE_RULES:
        if re.search(pat, name, re.I): return site
    return None

_cur = scored.drop_duplicates("Station_Code")[["Station_Code", "Station_Name", "State"]].copy()
_cur["site"] = [curated_site(n, s) for n, s in zip(_cur.Station_Name, _cur.State)]
_cur = _cur[_cur.site.notna()]
coords = pd.DataFrame({"Station_Code": _cur.Station_Code, "lat": _cur.site.map(lambda s: SAB_COORDS[s][0]), "lon": _cur.site.map(lambda s: SAB_COORDS[s][1]), "coord_source": "curated (approx.)"})

NOISE = re.compile(r"\b(U/S|D/S|U\.S|D\.S|B/C|B/F|H/W|NEAR|AFTER|BEFORE|MEETING|CONFLUENCE|CONF|WITH|OF|AND|WATER SUPPLY|INTAKE|WELL|POINT|SCHEME|DRINKING|VILLAGE|VILL|TALUKA|TAL|DIST|DISTT|DISTRICT|ROAD|BRIDGE|STATE|INTER|ON|TO|AT|IN|THE|NALLAH|NALA|RIVER|KHAD|GHAT|TOWN|CANT|CANTT|RAILWAY|STATION|CHOWKI|NH|PUL)\b\.?", re.I)
def place_candidates(name, river=""):
    """Up to 3 search phrases for a station, most specific first: the named place, the next place/district, then the last word (usually the district)."""
    s = name.upper(); paren = re.findall(r"\(([^)]*)\)", s); s = re.sub(r"\([^)]*\)", " ", s)
    parts = re.split(r"[,;]|\bAT\b|\bNEAR\b|\bAFTER\b|\bBEFORE\b|\bD/S\b|\bU/S\b| - ", s) + paren
    cands = []
    for p in parts:
        p = re.sub(r"[^A-Z ]", " ", NOISE.sub(" ", p)); p = re.sub(r"\s+", " ", p).strip()
        if len(p) >= 4 and p not in cands and _alpha(p) != _alpha(river): cands.append(p)
    out = cands[:2]
    if cands:
        last = cands[-1].split()[-1]
        if len(last) >= 5 and last not in out and last not in (river or "").upper(): out.append(last)
    return out[:3]

def geocode_query(q, state):
    """One Nominatim lookup restricted to India; the answer must lie in the station's state."""
    loc = _gc(q, country_codes="in", addressdetails=True)
    if not loc: return None
    if state and state != "INTER-STATE":
        st_found = (loc.raw.get("address", {}).get("state") or "").lower()
        if st_found and _alpha(state.title()) not in _alpha(st_found) and _alpha(st_found) not in _alpha(state.title()): return None
    return (loc.latitude, loc.longitude)

def geocode_stations(d, limit=None):
    global _gc
    from geopy.geocoders import Nominatim
    from geopy.extra.rate_limiter import RateLimiter
    f = "station_coords_cache.csv"
    cache = pd.read_csv(f) if os.path.exists(f) else pd.DataFrame(columns=["Station_Code", "lat", "lon", "coord_source"])
    todo = d.drop_duplicates("Station_Code")[["Station_Code", "Station_Name", "State", "River", "RiverGroup"]]
    todo = todo[~todo.Station_Code.isin(cache.Station_Code)]
    size = todo.groupby("RiverGroup").Station_Code.transform("size"); todo = todo.assign(_s=size).sort_values(["_s", "RiverGroup"], ascending=[False, True])   # biggest rivers first
    if limit: todo = todo.head(limit)
    _gc = RateLimiter(Nominatim(user_agent="jalnetra-sih25166-student-project", timeout=10).geocode, min_delay_seconds=1.1, error_wait_seconds=3, max_retries=1, swallow_exceptions=True)
    print(f"Geocoding {len(todo)} stations (≈ {len(todo) * 1.5 / 60:.0f} min). Press Stop to pause - progress is saved.")
    out, memo = [], {}
    def save():
        nonlocal cache
        cache = pd.concat([cache, pd.DataFrame(out, columns=["Station_Code", "lat", "lon", "coord_source"])]).drop_duplicates("Station_Code", keep="last")
        cache.to_csv(f, index=False); out.clear()
    try:
        for i, r in enumerate(todo.itertuples(), 1):
            hit, src = None, None
            for k, q in enumerate(place_candidates(r.Station_Name, r.River)):
                key = (q, r.State)
                if key not in memo: memo[key] = geocode_query(f"{q}, {r.State.title() if r.State else ''}, India", r.State)
                if memo[key]: hit, src = memo[key], ("geocoded (approx.)" if k == 0 else "geocoded (town/district-level, approx.)"); break
            out.append((r.Station_Code, *(hit or (nan, nan)), src))
            if i % 20 == 0: save(); print(f"  {i}/{len(todo)} done · located so far: {int(cache.lat.notna().sum())}")
    except KeyboardInterrupt:
        print("Paused - progress saved. Re-run this cell to continue.")
    save(); return cache

if GEOCODE_STATIONS:
    _d = scored if GEOCODE_YEARS == "all" else scored[scored.Year == scored.Year.max()]
    _d = _d[~_d.Station_Code.isin(coords.Station_Code)]
    gcache = geocode_stations(_d, GEOCODE_LIMIT).dropna(subset=["lat"])
    coords = pd.concat([coords, gcache[["Station_Code", "lat", "lon", "coord_source"]]])
if os.path.exists("station_coords_cache.csv") and not GEOCODE_STATIONS:
    _g = pd.read_csv("station_coords_cache.csv").dropna(subset=["lat"]); _g["coord_source"] = _g.get("coord_source", "geocoded (approx.)")
    coords = pd.concat([coords, _g[~_g.Station_Code.isin(coords.Station_Code)]])
if os.path.exists("verified_coordinates.csv"):
    vc = pd.read_csv("verified_coordinates.csv"); vc["coord_source"] = "verified"
    coords = pd.concat([vc[["Station_Code", "lat", "lon", "coord_source"]], coords])
    print(f"Loaded {len(vc)} verified coordinates")
coords = coords.drop_duplicates("Station_Code", keep="first").reset_index(drop=True)
print(coords.coord_source.value_counts().to_string() if len(coords) else "No coordinates yet (maps need them; everything else works).")

## 8 · River engine — every river
* **Station master**: one row per station code, with its river, state and years of data.
* **Order along the river**: located stations are ordered by the shortest path through their coordinates, starting from the station the PDF lists first; stations without coordinates keep the PDF's own order (the reports list stations roughly upstream → downstream).
* **River model** (rivers with ≥ 2 located sites): sites at the same place are merged into a yearly envelope; a river line through them (OpenStreetMap for rivers in `OSM_RIVERS`, otherwise straight lines) gives **km along the river**; any lat/long is snapped to it and **interpolated** (DO, pH, nitrate, temperature linearly; BOD, conductivity, coliform in log space).
* Every interpolated answer is labelled *interpolated*, with a confidence level and warnings.

In [ ]:
def hav(lat1, lon1, lat2, lon2):
    R = 6371.0088; p1, p2 = math.radians(lat1), math.radians(lat2); dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))

class RiverPath:
    """A polyline with cumulative distance; snap any lat/long to it and read positions back by chainage."""
    def __init__(self, pts):
        clean = [pts[0]] + [p for a, p in zip(pts, pts[1:]) if abs(p[0] - a[0]) + abs(p[1] - a[1]) > 1e-9]
        if len(clean) < 2: clean = [clean[0], (clean[0][0] + 1e-4, clean[0][1] + 1e-4)]
        self.pts = clean; cum = [0.0]
        for a, b in zip(clean, clean[1:]): cum.append(cum[-1] + hav(*a, *b))
        self.cum = np.array(cum); self.length = cum[-1]
        self.lat0, self.lon0 = np.mean([p[0] for p in clean]), np.mean([p[1] for p in clean])
        self.kx, self.ky = 111.32 * math.cos(math.radians(self.lat0)), 110.57
        self.xy = np.array([((p[1] - self.lon0) * self.kx, (p[0] - self.lat0) * self.ky) for p in clean])
    def snap(self, lat, lon):
        px, py = (lon - self.lon0) * self.kx, (lat - self.lat0) * self.ky
        A, B = self.xy[:-1], self.xy[1:]; d = B - A; L2 = np.maximum((d ** 2).sum(1), 1e-12)
        t = np.clip(((px - A[:, 0]) * d[:, 0] + (py - A[:, 1]) * d[:, 1]) / L2, 0, 1); proj = A + t[:, None] * d
        dist = np.hypot(proj[:, 0] - px, proj[:, 1] - py); i = int(dist.argmin())
        return float(dist[i]), float(self.cum[i] + t[i] * (self.cum[i + 1] - self.cum[i]))
    def at(self, c):
        c = float(np.clip(c, 0, self.length)); i = min(max(int(np.searchsorted(self.cum, c, side="right") - 1), 0), len(self.pts) - 2)
        t = (c - self.cum[i]) / max(self.cum[i + 1] - self.cum[i], 1e-12)
        return (self.pts[i][0] + t * (self.pts[i + 1][0] - self.pts[i][0]), self.pts[i][1] + t * (self.pts[i + 1][1] - self.pts[i][1]))
    def slice(self, c0, c1):
        c0, c1 = max(0, min(c0, c1)), min(self.length, max(c0, c1))
        return [self.at(c0)] + [self.pts[k] for k in range(len(self.pts)) if c0 < self.cum[k] < c1] + [self.at(c1)]

def fetch_osm_river(pattern, bbox):
    import urllib.request, urllib.parse
    b = f"({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]})"
    q = f'[out:json][timeout:90];(way["waterway"~"river|canal"]["name"~"{pattern}",i]{b};way["waterway"~"river|canal"]["name:en"~"{pattern}",i]{b};);out geom;'
    last = None
    for ep in ("https://overpass-api.de/api/interpreter", "https://overpass.kumi.systems/api/interpreter"):
        try:
            req = urllib.request.Request(ep, data=urllib.parse.urlencode({"data": q}).encode(), headers={"User-Agent": "jalnetra-sih25166"})
            data = json.load(urllib.request.urlopen(req, timeout=100))
            ways = [[(p["lat"], p["lon"]) for p in el["geometry"]] for el in data.get("elements", []) if el.get("geometry")]
            if ways: return ways
        except Exception as e: last = e
    print("   (OpenStreetMap download failed:", str(last)[:70], ")"); return None

def build_path_from_ways(ways, site_pts, labels=None, tol_km=5.0):
    G = nx.Graph(); key = lambda p: (round(p[0], 5), round(p[1], 5))
    for w in ways:
        for a, b in zip(w, w[1:]): G.add_edge(key(a), key(b), weight=hav(*a, *b))
    nodes = list(G.nodes); arr = np.array(nodes)
    def nearest(pt):
        d = np.hypot((arr[:, 0] - pt[0]) * 110.57, (arr[:, 1] - pt[1]) * 111.32 * math.cos(math.radians(pt[0]))); i = int(d.argmin()); return nodes[i], float(d[i])
    snapped = [nearest(p) for p in site_pts]; labels = labels or [str(i) for i in range(len(site_pts))]
    far = {labels[i]: round(d, 1) for i, (_, d) in enumerate(snapped) if d > tol_km}
    if far: print(f"   ⚠️ these coordinates are > {tol_km} km from the OSM river line - fix them: {far}")
    path, legs = [], []
    for k in range(len(site_pts) - 1):
        (n1, d1), (n2, d2) = snapped[k], snapped[k + 1]
        try:
            if max(d1, d2) > tol_km: raise nx.NetworkXNoPath
            seg = nx.shortest_path(G, n1, n2, weight="weight"); legs.append("river")
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            seg = [site_pts[k], site_pts[k + 1]]; legs.append("straight")
        path += seg if not path else seg[1:]
    return path, legs

def tsp_order(pts, start):
    """Open shortest path through points (nearest neighbour + 2-opt): upstream→downstream order for a river's stations."""
    n = len(pts)
    if n <= 2: return list(range(n))
    D = np.array([[hav(*pts[i], *pts[j]) for j in range(n)] for i in range(n)])
    path, left = [start], set(range(n)) - {start}
    while left: nxt = min(left, key=lambda j: D[path[-1], j]); path.append(nxt); left.remove(nxt)
    improved = True
    while improved:
        improved = False
        for i in range(1, n - 1):
            for j in range(i + 1, n):
                a, b = path[i - 1], path[i]; c = path[j]; d = path[j + 1] if j + 1 < n else None
                old = D[a, b] + (D[c, d] if d is not None else 0); new = D[a, c] + (D[b, d] if d is not None else 0)
                if new < old - 1e-9: path[i:j + 1] = reversed(path[i:j + 1]); improved = True
    return path

# ---- station master ----
scored["rank_pct"] = scored.groupby(["Year", "RiverGroup"]).ord.rank(pct=True)
master = (scored.sort_values("Year").groupby("Station_Code")
          .agg(Station_Name=("Station_Name", "last"), State=("State", "last"), River=("River", lambda s: s.mode().iloc[0]), RiverGroup=("RiverGroup", lambda s: s.mode().iloc[0]),
               first_year=("Year", "min"), last_year=("Year", "max"), n_years=("Year", "nunique"), seq=("rank_pct", "mean")).reset_index().merge(coords, on="Station_Code", how="left"))
master["located"] = master.lat.notna()
# geocoder mistakes: in a river with >= 5 located stations, drop any station > 250 km from its nearest sibling
for _rg, _g in master[master.located & ~master.RiverGroup.str.startswith("Other streams")].groupby("RiverGroup"):
    if len(_g) < 5: continue
    _p = list(zip(_g.lat, _g.lon))
    for _i, (_idx, _row) in enumerate(_g.iterrows()):
        if min(hav(*_p[_i], *_p[_j]) for _j in range(len(_p)) if _j != _i) > 250:
            master.loc[_idx, ["located", "coord_source"]] = [False, "dropped (implausible position)"]
print(f"{len(master)} stations · {master.RiverGroup.nunique()} river groups · {int(master.located.sum())} stations with coordinates")
RIVERS = (master.groupby("RiverGroup").agg(stations=("Station_Code", "size"), located=("located", "sum"), years=("n_years", "max"), state=("State", lambda s: s.mode().iloc[0] if len(s.mode()) else ""))
          .sort_values("stations", ascending=False))
display(RIVERS.head(25))

# ---- river models ----
MM_COLS = list(PARAM_COLS)
def envelope(d):
    return {c: (d[c].min() if c.endswith("_Min") else d[c].max()) for c in MM_COLS}

def build_model(river, osm=False):
    if river.startswith("Other streams"): return None            # unrelated streams do not form one river line
    m = master[(master.RiverGroup == river) & master.located]
    if len(m) < 2: return None
    m = m.assign(k=list(zip(m.lat.round(3), m.lon.round(3))))
    sites = m.groupby("k").agg(codes=("Station_Code", list), lat=("lat", "first"), lon=("lon", "first"), name=("Station_Name", lambda s: min(s, key=len)), seq=("seq", "min"), src=("coord_source", "first")).reset_index(drop=True)
    if len(sites) < 2: return None
    sites = sites.iloc[tsp_order(list(zip(sites.lat, sites.lon)), int(sites.seq.values.argmin()))].reset_index(drop=True)
    pts = list(zip(sites.lat, sites.lon)); mode = "straight lines between stations"; path_pts = pts
    if osm:
        print(f"Fetching the {river} line from OpenStreetMap …")
        ways = fetch_osm_river(river, (sites.lat.min() - .4, sites.lon.min() - .4, sites.lat.max() + .4, sites.lon.max() + .4))
        if ways:
            pp, legs = build_path_from_ways(ways, pts, list(sites.name))
            if "river" in legs: path_pts, mode = pp, f"OpenStreetMap river line ({legs.count('river')} of {len(legs)} legs)"
    path = RiverPath(path_pts)
    sites["chainage_km"] = [path.snap(a, b)[1] for a, b in zip(sites.lat, sites.lon)]
    if osm and list(sites.chainage_km) != sorted(sites.chainage_km):
        path = RiverPath(pts); mode = "straight lines (OSM order mismatch)"; sites["chainage_km"] = [path.snap(a, b)[1] for a, b in zip(sites.lat, sites.lon)]
    sites["label"] = [f"{n[:34]}" for n in sites.name]
    return {"river": river, "sites": sites, "path": path, "mode": mode, "codes": sum(sites.codes.tolist(), [])}

MODELS = {}
for _r in RIVERS.index:
    try:
        _m = build_model(_r, osm=_r in OSM_RIVERS)
        if _m: MODELS[_r] = _m
    except Exception as e: print("model failed for", _r, str(e)[:60])
print(f"River models (≥ 2 located sites): {len(MODELS)}", list(MODELS)[:12])

SITE_CACHE = {}
def site_table(river, year):
    key = (river, year)
    if key in SITE_CACHE: return SITE_CACHE[key]
    M = MODELS.get(river)
    if M is None: return pd.DataFrame()
    dy = scored[(scored.Year == year) & scored.Station_Code.isin(M["codes"])]
    rows = []
    for i, s in M["sites"].iterrows():
        d = dy[dy.Station_Code.isin(s.codes)]
        if d.empty: continue
        env = envelope(d); env.update(site=i, label=s.label, lat=s.lat, lon=s.lon, chainage_km=s.chainage_km, n_stations=len(d))
        for mode, pre in (("worst", "w_"), ("typical", "t_")):
            v = vector(pd.Series(env), mode)
            for p in PARAMS + ["pH_lo", "pH_hi"]: env[pre + p] = v[p]
        env["WQI_worst"] = wqi({k: env["w_" + k] for k in PARAMS + ["pH_lo", "pH_hi"]})
        rows.append(env)
    S = pd.DataFrame(rows)
    if len(S): S = S[S[["DO_Min", "BOD_Max"]].notna().any(axis=1)].sort_values("chainage_km").reset_index(drop=True)
    SITE_CACHE[key] = S
    return S

def model_years(river): return sorted(y for y in YEARS if len(site_table(river, y)))

def interp_at(S, chain, prefix):
    cs = S.chainage_km.values
    up = int(np.clip(np.searchsorted(cs, chain, side="right") - 1, 0, len(cs) - 1)); dn = min(up + 1, len(cs) - 1)
    t = 0.0 if dn == up or cs[dn] == cs[up] or chain <= cs[0] or chain >= cs[-1] else (chain - cs[up]) / (cs[dn] - cs[up])
    out = {}
    for p in PARAMS + ["pH_lo", "pH_hi"]:
        a, b = S[prefix + p].iloc[up], S[prefix + p].iloc[dn]
        if isn(a) and isn(b): out[p] = nan
        elif isn(a): out[p] = b
        elif isn(b): out[p] = a
        elif p in LOGP: out[p] = 10 ** ((1 - t) * math.log10(max(a, 1e-9)) + t * math.log10(max(b, 1e-9)))
        else: out[p] = (1 - t) * a + t * b
    return out, S.label.iloc[up], S.label.iloc[dn], up, dn

def query_location(river, lat, lon, year):
    M = MODELS[river]; S = site_table(river, year)
    if len(S) < 1: return None
    path = M["path"]; dist_river, chain = path.snap(lat, lon)
    d_all = [hav(lat, lon, r.lat, r.lon) for r in S.itertuples()]; ni = int(np.argmin(d_all)); d_near = d_all[ni]
    vw, upl, dnl, up, dn = interp_at(S, chain, "w_"); vt, *_ = interp_at(S, chain, "t_")
    measured = d_near <= 1.5
    if measured: vw = {k: S.loc[ni, "w_" + k] for k in vw}; vt = {k: S.loc[ni, "t_" + k] for k in vt}
    nearest_ch = np.abs(S.chainage_km.values - chain).min(); jump = 1.0
    if up != dn:
        a, b = S.loc[up, "w_BOD"], S.loc[dn, "w_BOD"]
        if not isn(a) and not isn(b): jump = max(a, b) / max(min(a, b), 1e-9)
    outside = len(S) < 2 or chain <= S.chainage_km.min() + 0.01 or chain >= S.chainage_km.max() - 0.01
    notes = []
    if measured: conf = "High (measured at station)"
    elif dist_river > 5 or outside: conf = "Low (outside the monitored river stretch)"
    elif nearest_ch < 8 and jump < 5: conf = "Medium"
    elif nearest_ch < 20 and jump < 10: conf = "Low-medium"
    else: conf = "Low"
    if jump >= 5 and not measured: notes.append(f"BOD changes {jump:.0f}× between {upl} and {dnl}: interpolation is only indicative")
    if outside: notes.append("Outside the monitored stretch: nearest station value held constant (extrapolation)")
    if dist_river > 5: notes.append(f"Point is {dist_river:.1f} km from the river line - results are for the nearest river point")
    if "approx" in str(M["sites"].src.iloc[0]): notes.append("Station coordinates are approximate, so positions along the river are approximate too")
    e = evaluate(vw, vt)
    return {"river": river, "lat": lat, "lon": lon, "year": year, "snap": path.at(chain), "dist_to_river_km": dist_river, "chainage_km": chain, "nearest_station": S.label.iloc[ni], "dist_nearest_km": d_near,
            "upstream": upl, "downstream": dnl, "source": "Measured" if measured else "Interpolated", "confidence": conf, "notes": notes, "v_worst": vw, "v_typ": vt, **e}

def locate_river(lat, lon, max_km=25):
    best = None
    for r, M in MODELS.items():
        d, _ = M["path"].snap(lat, lon)
        if best is None or d < best[1]: best = (r, d)
    return best[0] if best and best[1] <= max_km else None

if MODELS:
    _r = DEFAULT_RIVER if DEFAULT_RIVER in MODELS else next(iter(MODELS)); _y = model_years(_r)[-1]; _M = MODELS[_r]
    print(f"Self-test on {_r} ({_M['mode']}), year {_y}:")
    _q = query_location(_r, _M["sites"].lat.mean(), _M["sites"].lon.mean(), _y)
    print({k: (round(v, 1) if isinstance(v, float) else v) for k, v in _q.items() if k in ("source", "confidence", "WQI_worst", "CPCB_class", "Priority", "upstream", "downstream")})

## 9 · Maps and river charts (any river, any year)

In [ ]:
def popup_html(r):
    rows = "".join(f"<tr><td>{LABELS[p]}</td><td style='text-align:right'>{r[f'{COLMAP[p]}_Min']:g} – {r[f'{COLMAP[p]}_Max']:g}</td></tr>" for p in ["Temp", "DO", "pH", "EC", "BOD", "NO3", "FC", "TC"] if not isn(r[f"{COLMAP[p]}_Min"]))
    dark = r.WQI_cat in ("Good", "Poor", "Excellent")
    return (f"<div style='font-family:Arial;font-size:12px;width:310px'><b>{r.Station_Name}</b><br><span style='color:#666'>Code {r.Station_Code} · {r.State} · {int(r.Year)}</span>"
            f"<div style='margin:6px 0;padding:6px;background:{r.WQI_color};color:{'#000' if dark else '#fff'};border-radius:4px'>WQI (worst) <b>{fmt(r.WQI_worst, 0)}</b> – {r.WQI_cat}"
            f"<br>typical {fmt(r.WQI_typical, 0)} · CPCB class <b>{r.CPCB_class}</b> · Priority <b>{r.Priority}</b></div><table style='width:100%'>{rows}</table>"
            f"<div style='color:#888'>Min – Max during {int(r.Year)}</div></div>")

def fmt(x, d=1): return "–" if isn(x) else f"{x:.{d}f}"
LEGEND = lambda year: ("<div style='position:fixed;bottom:25px;left:25px;z-index:9999;background:white;padding:8px 10px;border:1px solid #999;border-radius:6px;font:12px Arial'>"
                       f"<b>WQI (worst-case) {year}</b><br>" + "".join(f"<span style='background:{c};display:inline-block;width:12px;height:12px;margin-right:6px'></span>{n}<br>" for _, n, c in WQI_BANDS) + "</div>")

def add_station_layers(m, d, light=False):
    d = d.merge(master[["Station_Code", "lat", "lon", "coord_source"]], on="Station_Code").dropna(subset=["lat"]).copy()
    d["cls"] = d.CPCB_class.map(CLASS_COL); d["pri"] = d.Priority.map(PRIORITY_COL)
    dup = d.duplicated(["lat", "lon"], keep="first"); d.loc[dup, ["lat", "lon"]] += 0.003
    layers = (("Stations · WQI category", "WQI_color", True),) if light else (("Stations · WQI category", "WQI_color", True), ("Stations · CPCB class", "cls", False), ("Stations · BOD priority", "pri", False))
    for name, col, show in layers:
        g = folium.FeatureGroup(name=name, show=show)
        for _, r in d.iterrows():
            lab = {"WQI_color": f"WQI {fmt(r.WQI_worst, 0)}", "cls": f"Class {r.CPCB_class}", "pri": f"Priority {r.Priority}"}[col]
            approx = "approx" in str(r.coord_source)
            folium.CircleMarker([r.lat, r.lon], radius=8, color="#222", weight=1.2, fill=True, fill_color=r[col], fill_opacity=0.35 if approx else 0.95, dash_array="3" if approx else None,
                                tooltip=f"{r.Station_Name} · {lab}" + (" (position approx.)" if approx else ""), popup=folium.Popup(popup_html(r), max_width=340, lazy=True) if not light else folium.Popup(f"<b>{r.Station_Name}</b><br>{r.River} · {r.State}<br>WQI {fmt(r.WQI_worst, 0)} · class {r.CPCB_class} · priority {r.Priority}<br>BOD max {fmt(r.BOD_Max)} · DO min {fmt(r.DO_Min)}", max_width=300)).add_to(g)
        g.add_to(m)
    return d

def finish_map(m, year):
    folium.LayerControl(collapsed=False).add_to(m)
    plugins.MiniMap(tile_layer=folium.TileLayer(tiles=ESRI["Esri Light Gray"], attr="Esri"), toggle_display=True).add_to(m)
    plugins.Fullscreen().add_to(m); plugins.MeasureControl().add_to(m); folium.LatLngPopup().add_to(m)
    m.get_root().html.add_child(folium.Element(LEGEND(year))); return m

def river_map(river, year):
    d = scored[(scored.Year == year) & scored.Station_Code.isin(master[master.RiverGroup == river].Station_Code)]
    loc = master[(master.RiverGroup == river) & master.located]
    if loc.empty: return None
    m = base_map([loc.lat.mean(), loc.lon.mean()], 8 if len(loc) > 2 else 11)
    m.fit_bounds([[loc.lat.min(), loc.lon.min()], [loc.lat.max(), loc.lon.max()]])
    S = site_table(river, year)
    if river in MODELS and len(S) >= 2:
        fg = folium.FeatureGroup(name=f"River quality {year} (interpolated WQI)", show=True); path = MODELS[river]["path"]
        c0, c1 = S.chainage_km.min(), S.chainage_km.max(); grid = np.arange(c0, c1, max(0.5, (c1 - c0) / 250))
        for a, b in zip(grid, list(grid[1:]) + [c1]):
            v, *_ = interp_at(S, (a + b) / 2, "w_"); w = wqi(v)
            folium.PolyLine(path.slice(a, b + 0.05), color=wqi_cat(w)[1], weight=7, opacity=0.9, tooltip=f"~{(a+b)/2:.0f} km · interpolated WQI {fmt(w, 0)}").add_to(fg)
        fg.add_to(m)
    add_station_layers(m, d); return finish_map(m, year)

def india_map(year):
    d = scored[scored.Year == year]
    m = base_map([22.5, 80.0], 5); add_station_layers(m, d, light=True); return finish_map(m, year)

# ---- river charts ----
def river_stations(river, year):
    """Stations of a river for a year in river order (model order if located, else the PDF order)."""
    d = scored[(scored.Year == year) & (scored.RiverGroup == river)].copy().merge(master[["Station_Code", "seq", "located"]], on="Station_Code", how="left")
    if river in MODELS:
        order = {c: (i, s.chainage_km) for i, s in MODELS[river]["sites"].iterrows() for c in s.codes}
        d["chain"] = d.Station_Code.map(lambda c: order.get(c, (nan, nan))[1]); d = d.sort_values(["chain", "ord"], na_position="last")
    else:
        d["chain"] = nan; d = d.sort_values("ord")
    d["short"] = d.Station_Name.str.replace(r"^(RIVER|R\.)\s+", "", regex=True).str[:34]
    return d.reset_index(drop=True)

def fig_river_profile(river, year):
    d = river_stations(river, year)
    if d.empty: return None
    x = d.chain if d.chain.notna().all() else pd.Series(range(1, len(d) + 1), index=d.index)
    xt = "km along the river (upstream → downstream)" if d.chain.notna().all() else "station order in the CPCB table (≈ upstream → downstream)"
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07, row_heights=[0.34, 0.33, 0.33], subplot_titles=("WQI, worst case (lower = cleaner)", "Dissolved oxygen min (mg/L)", "BOD max (mg/L, log)"))
    fig.add_bar(x=x, y=d.WQI_worst, marker_color=d.WQI_color, text=d.short, hovertemplate="%{text}<br>WQI %{y:.0f}<extra></extra>", showlegend=False, row=1, col=1)
    fig.add_trace(go.Scatter(x=x, y=d.DO_Min, mode="lines+markers", text=d.short, line=dict(color="#2c7fb8"), hovertemplate="%{text}<br>DO %{y}<extra></extra>", showlegend=False), 2, 1)
    fig.add_hline(y=5, line=dict(color="#c0392b", dash="dash", width=1), annotation_text="Class B ≥ 5", row=2, col=1)
    fig.add_trace(go.Scatter(x=x, y=d.BOD_Max, mode="lines+markers", text=d.short, line=dict(color="#d95f0e"), hovertemplate="%{text}<br>BOD %{y}<extra></extra>", showlegend=False), 3, 1)
    fig.add_hline(y=3, line=dict(color="#c0392b", dash="dash", width=1), annotation_text="limit 3", row=3, col=1); fig.update_yaxes(type="log", row=3, col=1)
    fig.update_xaxes(title_text=xt, row=3, col=1)
    fig.update_layout(height=700, title=f"{river} — water quality along the river, {year}", margin=dict(t=80)); return fig

def fig_river_trend(river):
    d = scored[scored.RiverGroup == river]
    g = d.groupby("Year").agg(median_WQI=("WQI_worst", "median"), worst_WQI=("WQI_worst", "max"), stations=("Station_Code", "nunique"), bad=("Priority", lambda s: 100 * s.isin(["I", "II", "III"]).mean()),
                              BOD=("BOD_Max", "median"), DO=("DO_Min", "median"))
    fig = make_subplots(rows=1, cols=3, subplot_titles=("WQI (median & worst station)", "% stations with BOD > 10 (Priority I–III)", "Median BOD max / DO min (mg/L)"))
    fig.add_trace(go.Scatter(x=g.index, y=g.median_WQI, name="median", mode="lines+markers"), 1, 1); fig.add_trace(go.Scatter(x=g.index, y=g.worst_WQI, name="worst station", mode="lines+markers", line=dict(color="#a50026")), 1, 1)
    fig.add_bar(x=g.index, y=g.bad, marker_color="#d95f0e", showlegend=False, row=1, col=2)
    fig.add_trace(go.Scatter(x=g.index, y=g.BOD, name="BOD max", mode="lines+markers", line=dict(color="#d95f0e")), 1, 3); fig.add_trace(go.Scatter(x=g.index, y=g.DO, name="DO min", mode="lines+markers", line=dict(color="#2c7fb8")), 1, 3)
    fig.update_xaxes(dtick=1); fig.update_layout(height=340, title=f"{river} — change over the years", margin=dict(t=80)); return fig

def fig_river_heat(river, maxn=45):
    d = scored[scored.RiverGroup == river].groupby(["Station_Code", "Year"]).WQI_worst.mean().unstack()
    nm = master.set_index("Station_Code").Station_Name.reindex(d.index).str.replace(r"^(RIVER|R\.)\s+", "", regex=True).str[:40]
    seq = master.set_index("Station_Code").seq.reindex(d.index); d = d.assign(_s=seq).sort_values("_s").drop(columns="_s").head(maxn); nm = nm.reindex(d.index)
    fig = go.Figure(go.Heatmap(z=d.values, x=[str(y) for y in d.columns], y=[f"{n} [{c}]" for n, c in zip(nm, d.index)], colorscale=[[0, "#1a9850"], [0.25, "#91cf60"], [0.5, "#fdae61"], [0.75, "#f46d43"], [1, "#a50026"]], zmin=0, zmax=120, colorbar=dict(title="WQI"), hovertemplate="%{y}<br>%{x}: WQI %{z:.0f}<extra></extra>"))
    fig.update_layout(title=f"{river} — WQI by station and year (blank = not monitored)", height=max(320, 22 * len(d) + 120), yaxis=dict(autorange="reversed"), margin=dict(l=300)); return fig

print(f"Stations on the map: {int(master.located.sum())} of {len(master)} ({master.RiverGroup[master.located].nunique()} river groups). Positions marked 'approx.' are drawn faded.")
india_map(int(LATEST))

In [ ]:
# One river at a time (stations coloured, river line coloured by interpolated WQI):
_rm = [r for r in MODELS if r != DEFAULT_RIVER and len(MODELS[r]["sites"]) >= 5][:1] or ([DEFAULT_RIVER] if DEFAULT_RIVER in MODELS else [])
if _rm: display(river_map(_rm[0], int(model_years(_rm[0])[-1])))

## 10 · Interactive dashboard
**River explorer** (any river, any year) · **Station** (any station in India, with its history) · **Location** (type a lat/long on any river that has located stations).

In [ ]:
def kpi(title, value, sub, color, text="#fff"):
    return (f"<div style='flex:1;min-width:150px;background:{color};color:{text};padding:10px 14px;border-radius:8px;font-family:Arial'>"
            f"<div style='font-size:11px;opacity:.85'>{title}</div><div style='font-size:26px;font-weight:700'>{value}</div><div style='font-size:11px'>{sub}</div></div>")
txtcol = lambda cat: "#000" if cat in ("Excellent", "Good", "Poor") else "#fff"
def kpi_row(e):
    cls = e["CPCB_class"]
    return ("<div style='display:flex;gap:10px;flex-wrap:wrap;margin:8px 0'>"
            + kpi("WQI · worst case", fmt(e["WQI_worst"], 0), e["WQI_cat"], wqi_cat(e["WQI_worst"])[1], txtcol(e["WQI_cat"]))
            + kpi("WQI · typical", fmt(e["WQI_typical"], 0), e["WQI_typ_cat"], wqi_cat(e["WQI_typical"])[1], txtcol(e["WQI_typ_cat"]))
            + kpi("CPCB best-use class", cls, (CLASS_DEF[cls][0][:42] if cls in CLASS_DEF else "Fails all classes" if cls == "Below E" else "no data"), CLASS_COL[cls], "#000" if cls in ("A", "B", "C", "D") else "#fff")
            + kpi("BOD priority", e["Priority"], "I = most urgent", PRIORITY_COL[e["Priority"]], "#000" if e["Priority"] in ("IV", "V", "Not polluted") else "#fff") + "</div>")

STD_TXT = {"DO": "≥ 5 (B) / ≥ 6 (A)", "BOD": "≤ 3", "pH": "6.5 – 8.5", "EC": "≤ 2250 (Class E)", "NO3": "≤ 10 (as N)", "FC": "≤ 500 (B)", "TC": "≤ 500 (B) / ≤ 5000 (C)", "Temp": "–"}
def param_table(vw, vt):
    def st(p, x):
        if isn(x): return "n/a"
        t = {"DO": x >= 5, "BOD": x <= 3, "pH": 6.5 <= x <= 8.5, "EC": x <= 2250, "NO3": x <= 10, "FC": x <= 500, "TC": x <= 5000, "Temp": True}[p]
        return "✅ within" if t else "❌ exceeds"
    return pd.DataFrame([{"Parameter": LABELS[p], "Typical": None if isn(vt[p]) else round(vt[p], 2), "Worst case": None if isn(vw[p]) else round(vw[p], 2), "Standard": STD_TXT[p], "Status (worst)": st(p, vw[p])}
                         for p in ["DO", "BOD", "pH", "EC", "NO3", "FC", "TC", "Temp"]])

def industry_block(vw):
    t = pd.DataFrame([{"Use (CPCB class)": f"{c} – {CLASS_DEF[c][0]}", "Suitable (worst case)": "✅ Yes" if CLASS_DEF[c][1](vw) else "❌ No"} for c in "ABCDE"])
    adv = []
    if not isn(vw["BOD"]) and vw["BOD"] > 3: adv.append("BOD above 3 mg/L: <b>no spare assimilative capacity</b> – any new organic discharge needs strict treatment / zero-liquid-discharge review.")
    if not isn(vw["DO"]) and vw["DO"] < 4: adv.append("DO falls below 4 mg/L at times: aquatic life is stressed; avoid discharges that add oxygen demand.")
    if not isn(vw["EC"]) and vw["EC"] > 2250: adv.append("High conductivity (salinity): unsuitable for irrigation or cooling-tower make-up without treatment; scaling/corrosion risk.")
    if not isn(vw["FC"]) and vw["FC"] > 500: adv.append("Fecal coliform above the bathing limit: disinfection needed before any human-contact use.")
    return t, adv or ["No major exceedance in the reported parameters."]
def show_html_list(items): display(HTML("<ul style='font-family:Arial;font-size:13px'>" + "".join(f"<li>{a}</li>" for a in items) + "</ul>"))

def station_trend_fig(hist):
    hist = hist.sort_values("Year")
    fig = make_subplots(rows=2, cols=3, subplot_titles=("WQI (lower = cleaner)", "DO range (mg/L)", "BOD range (mg/L, log)", "Conductivity range (log)", "Fecal coliform range (log)", "pH range"), vertical_spacing=0.18)
    x = hist.Year.astype(int).astype(str)
    fig.add_bar(x=x, y=hist.WQI_typical, marker_color="#5b8def", opacity=0.55, name="typical", row=1, col=1)
    fig.add_bar(x=x, y=hist.WQI_worst, marker_color=hist.WQI_color, name="worst", marker_line=dict(color="#222", width=1), row=1, col=1)
    for (r, c), p, lg, std in zip([(1, 2), (1, 3), (2, 1), (2, 2), (2, 3)], ["DO", "BOD", "Conductivity", "Fecal_Coliform", "pH"], [0, 1, 1, 1, 0], [5, 3, 2250, 500, 8.5]):
        for xi, a, b in zip(x, hist[f"{p}_Min"], hist[f"{p}_Max"]):
            if not (isn(a) and isn(b)): fig.add_trace(go.Scatter(x=[xi, xi], y=[a, b], mode="lines+markers", line=dict(color="#4a6fa5", width=4), marker=dict(size=7), showlegend=False), r, c)
        fig.add_hline(y=std, line=dict(color="#c0392b", dash="dash", width=1), row=r, col=c)
        if lg: fig.update_yaxes(type="log", row=r, col=c)
    fig.update_layout(height=620, showlegend=False, title="Year-by-year picture of this station (red dashed = standard)"); return fig

def show_station(code, year=None):
    hist = scored[scored.Station_Code == code].sort_values("Year")
    if hist.empty: print("No data for this station."); return
    year = year if year in set(hist.Year) else int(hist.Year.max()); r = hist[hist.Year == year].iloc[0]
    vw, vt = vector(r, "worst"), vector(r, "typical")
    display(HTML(f"<div style='font-family:Arial;font-size:13px'><h3 style='margin:4px 0'>{r.Station_Name}</h3>Code {code} · {r.State} · River: {r.River} · data years: {', '.join(str(int(y)) for y in hist.Year)} · showing <b>{year}</b></div>" + kpi_row(r)))
    display(param_table(vw, vt).style.hide(axis="index"))
    if len(hist) > 1: station_trend_fig(hist).show()
    h = hist[["Year", "WQI_worst", "WQI_cat", "CPCB_class", "Priority", "DO_Min", "BOD_Max", "Fecal_Coliform_Max"]].copy(); h["Year"] = h.Year.astype(int)
    display(HTML("<h4 style='font-family:Arial'>History</h4>")); display(h.round(1).style.hide(axis="index"))
    t, adv = industry_block(vw); display(HTML("<h4 style='font-family:Arial;margin-bottom:2px'>Industry / site screening</h4>")); display(t.style.hide(axis="index")); show_html_list(adv)
    ml = master[master.Station_Code == code].iloc[0]
    if ml.located:
        m = base_map([ml.lat, ml.lon], 11); folium.CircleMarker([ml.lat, ml.lon], radius=10, color="#222", fill=True, fill_color=r.WQI_color, fill_opacity=0.95, tooltip=r.Station_Name).add_to(m); display(m)
    else: print("(No coordinates for this station yet — see section 7.)")

def show_river(river, year):
    d = river_stations(river, year)
    if d.empty: print("No data for this river in", year); return
    worst = d.sort_values("WQI_worst", ascending=False).iloc[0]
    cls_ok = 100 * d.CPCB_class.isin(["A", "B", "C"]).mean()
    display(HTML(f"<div style='font-family:Arial;font-size:13px'><h3 style='margin:4px 0'>{river} — {year}</h3>{len(d)} stations · states: {', '.join(sorted(set(d.State) - {''}))} · "
                 f"{'river line: ' + MODELS[river]['mode'] if river in MODELS else 'no station coordinates → shown in CPCB table order'}</div>"
                 "<div style='display:flex;gap:10px;flex-wrap:wrap;margin:8px 0'>"
                 + kpi("Median WQI", fmt(d.WQI_worst.median(), 0), wqi_cat(d.WQI_worst.median())[0], wqi_cat(d.WQI_worst.median())[1], txtcol(wqi_cat(d.WQI_worst.median())[0]))
                 + kpi("Worst station", fmt(worst.WQI_worst, 0), worst.short, wqi_cat(worst.WQI_worst)[1], txtcol(wqi_cat(worst.WQI_worst)[0]))
                 + kpi("Stations meeting class A–C", f"{cls_ok:.0f}%", "fit for drinking after treatment or better", "#2c7fb8")
                 + kpi("Priority I–II stretches", int(d.Priority.isin(["I", "II"]).sum()), "BOD > 20 mg/L", "#a50026") + "</div>"))
    fig_river_profile(river, year).show(); fig_river_trend(river).show(); fig_river_heat(river).show()
    t = d[["Station_Code", "short", "State", "WQI_worst", "CPCB_class", "Priority", "DO_Min", "BOD_Max", "Conductivity_Max", "Fecal_Coliform_Max", "pH_Min", "pH_Max"]].rename(columns={"short": "Station"})
    display(HTML("<h4 style='font-family:Arial'>Stations (river order)</h4>")); display(t.round(1).style.hide(axis="index"))
    m = river_map(river, year)
    if m: display(m)
    else: print("(Map needs station coordinates — see section 7.)")

def show_point(lat, lon, year, river=None):
    river = river or locate_river(float(lat), float(lon))
    if river is None: print("No river with located stations within 25 km of this point. Provide coordinates (section 7) or pick a river in the explorer."); return
    if int(year) not in model_years(river): print(f"{river} has no located-station data for {year}; years available:", model_years(river)); return
    q = query_location(river, float(lat), float(lon), int(year)); wc = wqi_cat(q["WQI_worst"])[1]
    src = (f"<div style='font-family:Arial;font-size:13px'><h3 style='margin:4px 0'>{river}: point {lat:.4f}, {lon:.4f} — {q['source'].lower()} values, {year}</h3>"
           f"Confidence: <b>{q['confidence']}</b> · ~{q['chainage_km']:.0f} km along the monitored stretch, between <b>{q['upstream']}</b> and <b>{q['downstream']}</b> · nearest station {q['nearest_station']} ({q['dist_nearest_km']:.1f} km away)")
    for n in q["notes"]: src += f"<br>⚠️ {n}"
    display(HTML(src + "</div>" + kpi_row(q))); display(param_table(q["v_worst"], q["v_typ"]).style.hide(axis="index"))
    tr = []
    for y in model_years(river):
        qq = query_location(river, float(lat), float(lon), y); tr.append({"Year": y, "WQI worst": qq["WQI_worst"], "DO min": qq["v_worst"]["DO"], "BOD max": qq["v_worst"]["BOD"], "Class": qq["CPCB_class"], "Priority": qq["Priority"], "Source": qq["source"]})
    tr = pd.DataFrame(tr)
    if len(tr) > 1:
        f = make_subplots(rows=1, cols=3, subplot_titles=("WQI at this point", "DO min (mg/L)", "BOD max (mg/L)"))
        for k, c in enumerate(["WQI worst", "DO min", "BOD max"]): f.add_trace(go.Scatter(x=tr.Year, y=tr[c], mode="lines+markers", line=dict(color="#2c7fb8"), showlegend=False), 1, k + 1)
        f.update_xaxes(dtick=1); f.update_layout(height=320, title="This location across years"); f.show()
    display(HTML("<h4 style='font-family:Arial;margin-bottom:2px'>This location, every year</h4>")); display(tr.round(1).style.hide(axis="index"))
    t, adv = industry_block(q["v_worst"]); display(HTML("<h4 style='font-family:Arial;margin-bottom:2px'>Industry / site screening</h4>")); display(t.style.hide(axis="index")); show_html_list(adv)
    m = base_map(list(q["snap"]), 11); S = site_table(river, int(year))
    folium.CircleMarker([lat, lon], radius=7, color="#2c3e50", fill=True, fill_color="#fff", tooltip="Selected point").add_to(m)
    folium.CircleMarker(q["snap"], radius=10, color="#222", fill=True, fill_color=wc, fill_opacity=0.95, tooltip=f"WQI {fmt(q['WQI_worst'], 0)}").add_to(m)
    for s in S.itertuples(): folium.CircleMarker([s.lat, s.lon], radius=5, color="#444", fill=True, fill_color=wqi_cat(s.WQI_worst)[1], tooltip=s.label).add_to(m)
    display(m)

# ---------- widgets ----------
out_r, out_s, out_l = W.Output(), W.Output(), W.Output()
river_opts = [(f"{r}  ({int(row.stations)} stations{', map' if r in MODELS else ''})", r) for r, row in RIVERS.iterrows()]
w_river = W.Dropdown(options=river_opts, value=DEFAULT_RIVER if DEFAULT_RIVER in RIVERS.index else RIVERS.index[0], description="River:", layout=W.Layout(width="520px"))
w_ry = W.Dropdown(options=[int(y) for y in YEARS], value=int(LATEST), description="Year:", layout=W.Layout(width="150px"))
b_r = W.Button(description="Show river", button_style="primary")

states = ["All states"] + sorted(scored.State[scored.State != ""].unique())
w_state = W.Dropdown(options=states, description="State:", layout=W.Layout(width="320px"))
w_search = W.Text(placeholder="part of a river / station name", description="Search:", layout=W.Layout(width="420px"))
w_station = W.Dropdown(description="Station:", layout=W.Layout(width="800px"))
w_sy = W.Dropdown(options=[int(y) for y in YEARS], value=int(LATEST), description="Year:", layout=W.Layout(width="150px"))
b_s = W.Button(description="Show station", button_style="primary")
def refresh_station_list(*_):
    d = master.copy()
    if w_state.value != "All states": d = d[d.State == w_state.value]
    if w_search.value.strip(): d = d[d.Station_Name.str.contains(w_search.value.strip(), case=False, regex=False)]
    d = d.sort_values(["State", "Station_Name"]).head(400)
    w_station.options = [(f"{r.Station_Name[:80]}  [{r.Station_Code}] {r.State}", int(r.Station_Code)) for r in d.itertuples()] or [("— no match —", -1)]
w_state.observe(refresh_station_list, "value"); w_search.observe(refresh_station_list, "value"); refresh_station_list()
def _go_s(_):
    with out_s:
        clear_output(wait=True)
        if w_station.value not in (None, -1): show_station(int(w_station.value), int(w_sy.value))
def _go_r(_):
    with out_r: clear_output(wait=True); show_river(w_river.value, int(w_ry.value))
b_s.on_click(_go_s); b_r.on_click(_go_r)

loc_rivers = [("(auto-detect from lat/long)", "")] + [(r, r) for r in MODELS]
w_lr = W.Dropdown(options=loc_rivers, description="River:", layout=W.Layout(width="420px")); w_ly = W.Dropdown(options=[int(y) for y in YEARS], value=int(LATEST), description="Year:", layout=W.Layout(width="150px"))
w_lat = W.BoundedFloatText(value=23.05, min=6, max=37.5, step=0.001, description="Latitude"); w_lon = W.BoundedFloatText(value=72.60, min=68, max=98, step=0.001, description="Longitude")
b_l = W.Button(description="Show this location", button_style="success")
def _go_l(_):
    with out_l: clear_output(wait=True); show_point(w_lat.value, w_lon.value, w_ly.value, w_lr.value or None)
b_l.on_click(_go_l)
tab = W.Tab(children=[W.VBox([W.HBox([w_river, w_ry, b_r]), out_r]), W.VBox([W.HBox([w_state, w_search]), W.HBox([w_station, w_sy, b_s]), out_s]), W.VBox([W.HBox([w_lr, w_ly]), W.HBox([w_lat, w_lon, b_l]), out_l])])
for i, t_ in enumerate(("River explorer", "Any station", "Lat / long")): tab.set_title(i, t_)
display(HTML("<h3 style='font-family:Arial'>JalNetra dashboard</h3>")); display(tab)

### Same thing without the widgets

In [ ]:
_r = DEFAULT_RIVER if DEFAULT_RIVER in RIVERS.index else RIVERS.index[0]
show_river(_r, int(LATEST))

## 11 · Reports people can download
`report_station(code)` and `report_river(river, year)` write a **PDF**; `export_excel()` writes one **Excel workbook** for researchers (all years, all rivers). Only the data an industry user or researcher normally needs is included: the 8 key parameters (min/max), WQI, class, priority, standards check, usage suitability and trend.

In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors as rl
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import mm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak
os.makedirs("reports", exist_ok=True)
DISCLAIMER = ("Source: CPCB/NWMP annual river water-quality reports (min–max per year). WQI is a JalNetra index built on CPCB class limits and is not an official CPCB index. "
              "Interpolated values are estimates, not measurements. Verify before regulatory or legal use.")
def _pdf(x):
    return x.replace("✅ ", "").replace("❌ ", "").replace("µ", "u").replace("≥", ">=").replace("≤", "<=") if isinstance(x, str) else x
def _tbl(rows, widths=None, head=True, fs=7.5):
    rows = [[_pdf(c) for c in r] for r in rows]
    t = Table(rows, colWidths=widths, repeatRows=1 if head else 0)
    st = [("FONTSIZE", (0, 0), (-1, -1), fs), ("GRID", (0, 0), (-1, -1), 0.25, rl.grey), ("VALIGN", (0, 0), (-1, -1), "MIDDLE")]
    if head: st += [("BACKGROUND", (0, 0), (-1, 0), rl.HexColor("#e8eef5")), ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold")]
    t.setStyle(TableStyle(st)); return t
def _img(fig, name, w=170):
    f = f"reports/_{name}.png"; fig.savefig(f, dpi=140, bbox_inches="tight"); plt.close(fig)
    from PIL import Image as PI; iw, ih = PI.open(f).size; return Image(f, width=w * mm, height=w * mm * ih / iw)
def _cell(x, d=1): return "–" if isn(x) else (f"{x:.{d}f}" if isinstance(x, float) else str(x))

def report_station(code, path=None):
    hist = scored[scored.Station_Code == code].sort_values("Year")
    if hist.empty: raise ValueError("unknown station")
    r = hist.iloc[-1]; ss = getSampleStyleSheet(); path = path or f"reports/station_{code}.pdf"
    vw, vt = vector(r, "worst"), vector(r, "typical"); el = []
    el += [Paragraph(f"JalNetra — Station report", ss["Title"]), Paragraph(f"<b>{r.Station_Name}</b><br/>Code {code} · {r.State} · River {r.River} · latest data {int(r.Year)} · {len(hist)} year(s) of data", ss["Normal"]), Spacer(1, 6)]
    el += [_tbl([["WQI worst", "WQI typical", "CPCB class", "BOD priority"], [f"{_cell(r.WQI_worst, 0)} ({r.WQI_cat})", f"{_cell(r.WQI_typical, 0)} ({r.WQI_typ_cat})", f"{r.CPCB_class}", f"{r.Priority}"]], fs=9), Spacer(1, 6)]
    pt = param_table(vw, vt); el += [Paragraph(f"Parameters, {int(r.Year)}", ss["Heading3"]), _tbl([list(pt.columns)] + [[("" if isn(v) else v) for v in row] for row in pt.values.tolist()], [55 * mm, 22 * mm, 22 * mm, 38 * mm, 30 * mm]), Spacer(1, 6)]
    if len(hist) > 1:
        fig, ax = plt.subplots(1, 4, figsize=(11, 2.6)); yy = hist.Year.astype(int)
        for a, (c, lab, lg, std) in zip(ax, [("WQI_worst", "WQI (worst)", 0, None), ("DO_Min", "DO min (mg/L)", 0, 5), ("BOD_Max", "BOD max (mg/L)", 1, 3), ("Fecal_Coliform_Max", "Fecal coliform max", 1, 500)]):
            a.plot(yy, hist[c], "o-", color="#2c7fb8"); a.set_title(lab, fontsize=8); a.tick_params(labelsize=7); a.set_xticks(yy)
            if std: a.axhline(std, color="#c0392b", ls="--", lw=1)
            if lg: a.set_yscale("log")
        el += [Paragraph("Trend", ss["Heading3"]), _img(fig, f"s{code}"), Spacer(1, 6)]
    h = hist[["Year", "WQI_worst", "CPCB_class", "Priority", "DO_Min", "BOD_Max", "Conductivity_Max", "Fecal_Coliform_Max"]]
    el += [Paragraph("History", ss["Heading3"]), _tbl([["Year", "WQI", "Class", "Priority", "DO min", "BOD max", "EC max", "FC max"]] + [[int(a), _cell(b, 0), c, d_, _cell(e), _cell(f), _cell(g, 0), _cell(h_, 0)] for a, b, c, d_, e, f, g, h_ in h.itertuples(index=False)])]
    t, adv = industry_block(vw)
    el += [Spacer(1, 6), Paragraph("Industry / site screening (worst case)", ss["Heading3"]), _tbl([list(t.columns)] + t.values.tolist(), [130 * mm, 40 * mm])]
    el += [Spacer(1, 4)] + [Paragraph("• " + re.sub("<[^>]+>", "", a), ss["Normal"]) for a in adv] + [Spacer(1, 10), Paragraph(DISCLAIMER, ss["Italic"])]
    SimpleDocTemplate(path, pagesize=A4, leftMargin=15 * mm, rightMargin=15 * mm, topMargin=14 * mm, bottomMargin=14 * mm).build(el); return path

def report_river(river, year=None, path=None):
    year = year or int(scored[scored.RiverGroup == river].Year.max()); d = river_stations(river, year)
    if d.empty: raise ValueError("no data")
    ss = getSampleStyleSheet(); path = path or f"reports/river_{re.sub('[^A-Za-z0-9]+', '_', river)}_{year}.pdf"; el = []
    el += [Paragraph(f"JalNetra — River report: {river}, {year}", ss["Title"]),
           Paragraph(f"{len(d)} stations · states: {', '.join(sorted(set(d.State) - {''}))} · median WQI {_cell(d.WQI_worst.median(), 0)} · worst station WQI {_cell(d.WQI_worst.max(), 0)} · "
                     f"stations in Priority I–II (BOD &gt; 20): {int(d.Priority.isin(['I', 'II']).sum())} · meeting class A–C: {100 * d.CPCB_class.isin(['A', 'B', 'C']).mean():.0f}%", ss["Normal"]), Spacer(1, 6)]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2)); x = range(1, len(d) + 1)
    ax[0].bar(x, d.WQI_worst, color=d.WQI_color); ax[0].set_title("WQI by station (river order →)", fontsize=9); ax[0].tick_params(labelsize=7)
    ax[1].plot(x, d.DO_Min, "o-", label="DO min", color="#2c7fb8"); ax[1].plot(x, d.BOD_Max, "s-", label="BOD max", color="#d95f0e"); ax[1].set_yscale("log"); ax[1].axhline(3, color="#c0392b", ls="--", lw=1); ax[1].legend(fontsize=7); ax[1].tick_params(labelsize=7); ax[1].set_title("DO and BOD along the river", fontsize=9)
    el += [_img(fig, "r_prof")]
    g = scored[scored.RiverGroup == river].groupby("Year").agg(m=("WQI_worst", "median"), w=("WQI_worst", "max"), n=("Station_Code", "nunique"), bad=("Priority", lambda s: 100 * s.isin(["I", "II", "III"]).mean()))
    if len(g) > 1:
        fig, ax = plt.subplots(1, 2, figsize=(11, 2.6)); ax[0].plot(g.index, g.m, "o-", label="median"); ax[0].plot(g.index, g.w, "s-", color="#a50026", label="worst"); ax[0].legend(fontsize=7); ax[0].set_title("WQI over the years", fontsize=9); ax[0].set_xticks(g.index)
        ax[1].bar(g.index, g.bad, color="#d95f0e"); ax[1].set_title("% stations with BOD > 10 mg/L", fontsize=9); ax[1].set_xticks(g.index)
        for a in ax: a.tick_params(labelsize=7)
        el += [Spacer(1, 4), _img(fig, "r_trend")]
    top = d.sort_values("WQI_worst", ascending=False).head(15)
    el += [Paragraph("Most polluted stations", ss["Heading3"]), _tbl([["Code", "Station", "WQI", "Class", "Priority", "DO min", "BOD max", "FC max"]] + [[int(r.Station_Code), r.short[:38], _cell(r.WQI_worst, 0), r.CPCB_class, r.Priority, _cell(r.DO_Min), _cell(r.BOD_Max), _cell(r.Fecal_Coliform_Max, 0)] for r in top.itertuples()], [14 * mm, 70 * mm, 12 * mm, 14 * mm, 16 * mm, 16 * mm, 16 * mm, 18 * mm])]
    el += [PageBreak(), Paragraph("All stations (river order)", ss["Heading3"]), _tbl([["Code", "Station", "WQI", "Cls", "Pri", "DO", "BOD", "EC", "FC", "pH"]] + [[int(r.Station_Code), r.short[:34], _cell(r.WQI_worst, 0), r.CPCB_class, r.Priority, _cell(r.DO_Min), _cell(r.BOD_Max), _cell(r.Conductivity_Max, 0), _cell(r.Fecal_Coliform_Max, 0), f"{_cell(r.pH_Min)}–{_cell(r.pH_Max)}"] for r in d.itertuples()], [14 * mm, 62 * mm, 11 * mm, 10 * mm, 12 * mm, 12 * mm, 12 * mm, 14 * mm, 16 * mm, 20 * mm], fs=6.5), Spacer(1, 8), Paragraph(DISCLAIMER, ss["Italic"])]
    SimpleDocTemplate(path, pagesize=A4, leftMargin=15 * mm, rightMargin=15 * mm, topMargin=14 * mm, bottomMargin=14 * mm).build(el); return path

ESSENTIAL = ["Year", "Station_Code", "Station_Name", "State", "River", "RiverGroup", "Temp_Min", "Temp_Max", "DO_Min", "DO_Max", "pH_Min", "pH_Max", "Conductivity_Min", "Conductivity_Max", "BOD_Min", "BOD_Max",
             "Nitrate_Min", "Nitrate_Max", "Fecal_Coliform_Min", "Fecal_Coliform_Max", "Total_Coliform_Min", "Total_Coliform_Max", "WQI_worst", "WQI_typical", "WQI_cat", "CPCB_class", "Priority"]
def export_excel(path="jalnetra_data.xlsx"):
    pr = scored.copy()
    pr["Meets_BOD<=3"] = pr.BOD_Max <= 3; pr["Meets_DO>=5"] = pr.DO_Min >= 5; pr["Meets_pH_6.5-8.5"] = (pr.pH_Min >= 6.5) & (pr.pH_Max <= 8.5); pr["Meets_FC<=500"] = pr.Fecal_Coliform_Max <= 500
    pv = scored.pivot_table(index="Station_Code", columns="Year", values="WQI_worst", aggfunc="mean"); pv.columns = [f"WQI_{int(c)}" for c in pv.columns]
    st = master[["Station_Code", "Station_Name", "State", "River", "RiverGroup", "first_year", "last_year", "n_years", "lat", "lon", "coord_source"]].merge(pv.reset_index(), on="Station_Code", how="left")
    rv = scored.groupby(["RiverGroup", "Year"]).agg(stations=("Station_Code", "nunique"), median_WQI=("WQI_worst", "median"), worst_WQI=("WQI_worst", "max"), median_BOD_max=("BOD_Max", "median"), median_DO_min=("DO_Min", "median"),
                                                   pct_priority_I_III=("Priority", lambda s: round(100 * s.isin(["I", "II", "III"]).mean(), 1))).reset_index()
    last = scored[scored.Year == LATEST]; pri = last[last.Priority.isin(["I", "II", "III"])].sort_values("BOD_Max", ascending=False)[["Station_Code", "Station_Name", "State", "River", "BOD_Max", "DO_Min", "Priority", "WQI_worst"]]
    std = pd.DataFrame([{"Parameter": LABELS[p], "Standard / class limit": STD_TXT[p]} for p in ["DO", "BOD", "pH", "EC", "NO3", "FC", "TC"]])
    notes = pd.DataFrame({"Notes": [DISCLAIMER, "Min/Max are the lowest and highest values reported for the year; 'worst case' uses DO min and the maxima of other parameters.", "Coordinates marked 'approx.' are estimates."]})
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        pr[ESSENTIAL + ["Meets_BOD<=3", "Meets_DO>=5", "Meets_pH_6.5-8.5", "Meets_FC<=500"]].to_excel(xw, sheet_name="Readings (all years)", index=False)
        st.to_excel(xw, sheet_name="Stations", index=False); rv.to_excel(xw, sheet_name="River summary", index=False); pri.to_excel(xw, sheet_name=f"Priority I-III {LATEST}", index=False)
        std.to_excel(xw, sheet_name="Standards", index=False); notes.to_excel(xw, sheet_name="Notes", index=False)
    return path

_c = int(master.sort_values("n_years", ascending=False).Station_Code.iloc[0]); print("Sample station report:", report_station(_c))
print("Sample river report:", report_river(_r, int(LATEST)))
print("Excel:", export_excel())

## 12 · Website data pack (`web/` folder)
Writes `web/data.json` — compact, only what the site needs — and a ready-made **front end** (`web/index.html`) that reads it. Open the folder with any static host (GitHub Pages, Netlify, Cloudflare Pages) or locally with `python -m http.server`.

In [ ]:
os.makedirs("web", exist_ok=True)
rd = lambda x, n=2: None if isn(x) else round(float(x), n)
R_COLS = ["Temp_Min", "Temp_Max", "DO_Min", "DO_Max", "pH_Min", "pH_Max", "Conductivity_Min", "Conductivity_Max", "BOD_Min", "BOD_Max", "Nitrate_Min", "Nitrate_Max", "Fecal_Coliform_Min", "Fecal_Coliform_Max", "Total_Coliform_Min", "Total_Coliform_Max"]
rivers_list = list(RIVERS.index); ridx = {r: i for i, r in enumerate(rivers_list)}
states_list = sorted(set(master.State) - {""}); sidx = {s: i for i, s in enumerate(states_list)}
stations_js = [{"c": int(r.Station_Code), "n": r.Station_Name, "s": sidx.get(r.State, -1), "r": ridx[r.RiverGroup], "lat": rd(r.lat, 5), "lon": rd(r.lon, 5), "q": ("v" if r.coord_source == "verified" else "a") if r.located else None,
               "o": rd(r.seq, 4)} for r in master.itertuples()]
CLS = ["A", "B", "C", "D", "E", "Below E", "No data"]; PRI = ["I", "II", "III", "IV", "V", "Not polluted", "No data"]
readings_js = [[int(r.Year), int(r.Station_Code)] + [rd(getattr(r, c)) for c in R_COLS] + [rd(r.WQI_worst, 1), rd(r.WQI_typical, 1), CLS.index(r.CPCB_class), PRI.index(r.Priority)] for r in scored.itertuples()]
rivers_js = [{"name": r, "model": r in MODELS, "path": ([[round(p[0], 4), round(p[1], 4)] for p in MODELS[r]["path"].pts[::max(1, len(MODELS[r]["path"].pts) // 400)]] if r in MODELS else None)} for r in rivers_list]
web = {"meta": {"title": "JalNetra", "years": [int(y) for y in YEARS], "latest": int(LATEST), "source": "CPCB / NWMP annual river water-quality reports", "disclaimer": DISCLAIMER},
       "r_cols": R_COLS, "states": states_list, "rivers": rivers_js, "classes": CLS, "priorities": PRI, "stations": stations_js, "readings": readings_js}
json.dump(web, open("web/data.json", "w"), separators=(",", ":"))
print(f"web/data.json: {os.path.getsize('web/data.json')/1e6:.1f} MB · {len(stations_js)} stations · {len(readings_js)} readings · {len(rivers_js)} river groups")

In [ ]:
export_cols = ESSENTIAL
scored[export_cols].to_csv("jalnetra_all_years_clean.csv", index=False)
master.to_csv("jalnetra_stations.csv", index=False); rejects.to_csv("parse_rejects.csv", index=False)
OUT = ["jalnetra_all_years_clean.csv", "jalnetra_stations.csv", "jalnetra_data.xlsx", "web/data.json", "parse_rejects.csv"]
print("Saved:", ", ".join(OUT))
try:
    import shutil; shutil.make_archive("jalnetra_web", "zip", "web")
    from google.colab import files
    for f in OUT[:3] + ["jalnetra_web.zip"]: files.download(f)
except ImportError:
    pass

## 13 · Next
1. Put official coordinates in `verified_coordinates.csv` (`Station_Code, lat, lon`); rerun sections 7–8.
2. Download `jalnetra_web.zip`, unzip, drop `index.html` (supplied separately) next to `data.json` and host it.
3. Add the 2023 report when it is available — drop the PDF in and delete `parsed_cache.csv`.